# ASL fingerspelling: train and evaluate the TCN + CTC model on FSboard

**What this notebook does.** It teaches a small network to read fingerspelled words and short phrases from video,
letter by letter and with spaces between words, in order. Everything happens here in Google Colab. Nothing from the
dataset goes on your laptop.

| Step | What happens |
|---|---|
| 1 | Check the GPU and connect Google Drive |
| 2 | Choose your data: one-word clips and phrases with spaces |
| 3 | Run MediaPipe on the videos to get **all 21 hand points** |
| 4 | Turn the points into model inputs and check them |
| 5 | Optional tiny test: memorise a few clips, to prove the pipeline works |
| 6 | Train the model (speed range, longer memory, start from your earlier model) |
| 7 | Evaluate it on signers it has never seen, next to your earlier model |
| 8 | Speed check: the same test clips signed faster and slower |
| 9 | Export a small model file |
| 10 | Optional: fine-tune on your own webcam recordings (off unless you switch it on) |

**Why videos?** FSboard's ready-made landmark files hold only 8 of the 21 hand points, so we make our own.

### How to use it
1. Upload this one file at [colab.research.google.com](https://colab.research.google.com) (**File → Upload notebook**).
2. **Runtime → Change runtime type → T4 GPU.**
3. **Runtime → Run all.** A Google pop-up asks to connect your Drive.
4. The settings are those of the full run. Step 3 can take hours the first time; the progress lines estimate it.

### If Colab disconnects, or in a new session
Colab forgets every variable when a session ends. Step 4 saves its result to Drive, so you don't repeat steps 2–4:
run **Setup** and **step 1**, then go straight to the step you want (5 to 10). It prints which run it loaded.
Run steps 2, 3 and 4 again only when you change which clips to use. They need no GPU, so a CPU runtime saves your limit.
Training continues from the last saved epoch.

### Where files go
| Place | What | How long it lasts |
|---|---|---|
| **Temporary Colab disk** | code, metadata, MediaPipe, one video at a time | erased when the session ends |
| **Google Drive** `MyDrive/asl_fingerspelling/fsboard_tcn_ctc/` | hand points, checkpoints, best models, reports, zips | until you delete it |
| **Your laptop** | nothing, unless you choose to download a zip | |

Earlier runs on your Drive are never changed: every new setting gets its own run folder.

**Please keep the videos private.** FSboard's rules ask you to blur signers' faces when publicizing examples. This
notebook never displays a video. Credit: FSboard by Georg et al., CC BY 4.0
([paper](https://arxiv.org/abs/2407.15806), [dataset](https://www.kaggle.com/datasets/googleai/fsboard)).

In [ ]:
#@title Setup: run this first (the code stays hidden; click "Show code" to read it)
# Writes the project's code to the temporary Colab disk. These are exact copies
# of the files named below, from backend/ in the repository.
import importlib, json, os, shutil, sys
from pathlib import Path

WORK = Path(os.environ.get("FSBOARD_WORK", "/content/fsboard_tcn"))   # temporary Colab disk
CODE = WORK / "code"

_FILES = {
    "asl/__init__.py": r'''"""
asl - core package for the ASL fingerspelling project.

Modules
-------
features      Landmark preprocessing (identical to the original utils.py) + version tag.
tracker       MediaPipe hand tracking wrapper (supporting tool, not the recogniser).
classifier    Loads OUR trained TensorFlow model and returns letter probabilities.
text_builder  Turns per-frame predictions into accepted letters / words (pure logic).
engine        One live recognition session: JPEG frame -> prediction -> text update.
registry      Versioned models (model + label map + preprocessing) and the active model.
dataset       SQLite metadata + local media storage for imported training videos.
training      Grouped train/val/test split, retraining, comparison with the baseline.
speech        Optional local speech-to-text (Vosk). Never required for recognition.

The original baseline (train.py, app.py, ...) is in baseline/ and still works
on its own.
"""
from pathlib import Path

# Where everything lives. This file is <project>/backend/asl/__init__.py
BACKEND_DIR = Path(__file__).resolve().parent.parent
ROOT = BACKEND_DIR.parent                         # the project folder
MODELS_DIR = BACKEND_DIR / "models"               # registry, retrained versions, speech model
STORAGE_DIR = BACKEND_DIR / "storage"             # imported videos and their database
BASELINE_DIR = ROOT / "baseline"                  # original scripts
BASELINE_MODEL_DIR = BASELINE_DIR / "model"       # original model and label map (protected)
FRONTEND_DIST = ROOT / "frontend" / "dist"        # built browser interface
''',
    "asl/features.py": r'''"""
features.py - Landmark preprocessing shared by live recognition, video
extraction and training.

This is deliberately the SAME maths as utils.landmarks_to_features() in the
original project, so the existing trained model (baseline/model/asl_mlp.keras) keeps
working unchanged. tests/test_pipeline.py checks that both give identical
numbers.

Every extracted sample and every trained model stores PREPROCESSING_VERSION.
If the maths here ever changes, bump the version so old features and old
models are never silently mixed with new ones.
"""
import numpy as np

NUM_LANDMARKS = 21
NUM_FEATURES = NUM_LANDMARKS * 3

# Version tag stored next to features and models.
PREPROCESSING_VERSION = "landmarks-v1"

# Human-readable description written into each model version folder.
PREPROCESSING = {
    "version": PREPROCESSING_VERSION,
    "num_features": NUM_FEATURES,
    "hand_tracker": "mediapipe.solutions.hands (0.10.14), max_num_hands=1",
    "input": "camera frame mirrored horizontally before tracking (same as original app.py)",
    "normalisation": [
        "subtract wrist landmark (index 0) so the wrist is the origin",
        "divide by the largest 2D distance from the wrist (z ignored for scale)",
        "flatten (21, 3) -> (63,) as x0,y0,z0,x1,y1,z1,...",
    ],
    "train_augmentation": "mirror copy (x -> -x) added to TRAINING split only, after splitting",
}


def normalise_points(points):
    """Normalise raw landmark points.

    points: array-like of shape (21, 3) with MediaPipe x, y (0..1 image
            coordinates) and z (relative depth).
    returns: float32 array of shape (63,)
    """
    pts = np.array(points, dtype=np.float32).reshape(NUM_LANDMARKS, 3).copy()
    pts -= pts[0]                                         # wrist -> origin
    scale = np.max(np.linalg.norm(pts[:, :2], axis=1))    # hand "size" in 2D
    if scale > 0:
        pts /= scale
    return pts.flatten()


def points_from_mediapipe(hand_landmarks):
    """MediaPipe NormalizedLandmarkList -> numpy (21, 3) raw points."""
    return np.array([[lm.x, lm.y, lm.z] for lm in hand_landmarks.landmark],
                    dtype=np.float32)


def hand_size(points):
    """Largest 2D wrist-to-landmark distance in raw image units (0..1).

    Used to measure hand movement relative to the hand's own size, so the
    same rule works whether the hand is near or far from the camera.
    """
    pts = np.asarray(points, dtype=np.float32).reshape(NUM_LANDMARKS, 3)
    return float(np.max(np.linalg.norm(pts[:, :2] - pts[0, :2], axis=1)))


def mirror(features):
    """Flip feature vector(s) left <-> right by negating every x value."""
    f = np.array(features, dtype=np.float32, copy=True)
    f[..., 0::3] *= -1
    return f
''',
    "asl/text_builder.py": r'''"""
text_builder.py - Turns per-frame model predictions into accepted text.

Pure Python, no camera and no TensorFlow, so it can be unit-tested with
made-up prediction streams (see tests/test_text_builder.py).

How a letter is accepted
------------------------
Every frame gives (hand present?, predicted label, confidence).

1. Frames with confidence < min_confidence count as "uncertain" and vote for
   nothing. They never add a letter.
2. A letter is ACCEPTED when, over the last `hold_seconds`, the hand was
   visible the whole time, at least `min_frames` frames were seen, and the
   same label won at least `agreement` of them.
3. After a letter is accepted it is "blocked": holding the same handshape
   does NOT add it again.

How a deliberate repeated letter works (e.g. the LL in HELLO, OO in BOOK)
------------------------------------------------------------------------
The blocked letter is "re-armed" by any one of:
  a) dropping the hand out of view for at least `rearm_absent_seconds`;
  b) relaxing / changing the handshape so that the accepted letter is not
     the confident prediction for `rearm_other_seconds`;
  c) sliding the hand sideways by `rearm_move_hand_sizes` hand widths while
     holding the letter (in ASL, double letters are usually shown with a
     small slide or bounce).
After re-arming, the letter needs a fresh full hold before it is added.

Word spacing
------------
* Hand out of view for `auto_space_seconds` -> one space (0 disables it).
* The model's `space` / `del` control gestures, or the UI buttons.

Nothing here guesses words or rewrites text. The user corrects text with
set_text() (the Edit box in the interface).
"""
from collections import deque
from dataclasses import dataclass, asdict, field
from typing import Optional

CONTROL_LABELS = {"space", "del"}   # dataset control gestures, not ASL letters


@dataclass
class BuilderConfig:
    min_confidence: float = 0.80
    hold_seconds: float = 0.45
    min_frames: int = 4
    agreement: float = 0.70
    rearm_absent_seconds: float = 0.15
    rearm_other_seconds: float = 0.35
    rearm_move_hand_sizes: float = 0.60   # 0 disables movement re-arm
    auto_space_seconds: float = 1.2       # 0 disables automatic spaces
    control_gestures: bool = True         # obey model's space/del gestures

    def update(self, **kw):
        for k, v in kw.items():
            if not hasattr(self, k):
                raise KeyError(f"Unknown setting: {k}")
            setattr(self, k, type(getattr(self, k))(v))

    def as_dict(self):
        return asdict(self)


@dataclass
class FrameResult:
    """What the interface needs to show for one processed frame."""
    state: str                      # no_hand | uncertain | holding | accepted
    label: Optional[str] = None     # top prediction this frame
    confidence: float = 0.0
    candidate: Optional[str] = None # letter currently being held
    progress: float = 0.0           # 0..1 towards acceptance
    blocked: bool = False           # candidate already accepted, waiting re-arm
    accepted: Optional[str] = None  # label accepted on THIS frame
    events: list = field(default_factory=list)


class TextBuilder:
    def __init__(self, config: Optional[BuilderConfig] = None):
        self.cfg = config or BuilderConfig()
        self.text = ""                 # sentence being built
        self.sentences = []            # finished sentences
        self.sentence_finished = False # True right after Finish sentence
        self._event_id = 0
        self._reset_tracking()

    # ------------------------------------------------------------------ state
    def _reset_tracking(self):
        self._window = deque()         # (t, label or None)
        self._hand_since = None        # time the hand became visible
        self._absent_since = None      # time the hand disappeared
        self._last_commit_t = None
        self._blocked = None           # label accepted and not yet re-armed
        self._blocked_last_seen = None # last time blocked label was confident
        self._commit_wrist = None      # wrist position at acceptance
        self._space_added_for_absence = False

    def _event(self, kind, value=None):
        self._event_id += 1
        ev = {"id": self._event_id, "type": kind, "value": value, "text": self.text}
        if kind == "finish":
            ev["finished_text"] = value
        return ev

    def snapshot(self):
        return {"text": self.text, "sentences": list(self.sentences),
                "sentence_finished": self.sentence_finished}

    # ------------------------------------------------------- text operations
    def _append_letter(self, letter, events):
        self._leave_finished_state()
        self.text += letter
        events.append(self._event("letter", letter))

    def _append_space(self, events, kind="space"):
        if self.text and not self.text.endswith(" "):
            self.text += " "
            events.append(self._event(kind, " "))

    def add_space(self):
        ev = []
        self._append_space(ev)
        return ev

    def delete(self):
        ev = []
        if self.text:
            removed = self.text[-1]
            self.text = self.text[:-1]
            ev.append(self._event("delete", removed))
        return ev

    def clear(self):
        self.text = ""
        self.sentence_finished = False
        return [self._event("clear")]

    def set_text(self, text):
        """User correction from the Edit box. Kept exactly as typed (upper-cased)."""
        self._leave_finished_state()
        self.text = " ".join(str(text).upper().split()) + (" " if str(text).endswith(" ") else "")
        return [self._event("edit", self.text)]

    def finish(self):
        """Finish sentence: store it, start a new one, emit ONE finish event."""
        finished = self.text.strip()
        if finished:
            self.sentences.append(finished)
        self.text = ""
        self.sentence_finished = True
        self._blocked = None   # a new sentence may start with the same letter
        return [self._event("finish", finished)]

    def _leave_finished_state(self):
        self.sentence_finished = False

    # ----------------------------------------------------------- per frame
    def update(self, t, label=None, confidence=0.0, wrist=None, size=None):
        """Feed one frame.

        t          timestamp in seconds (monotonic)
        label      top predicted label, or None if no hand was detected
        confidence probability of that label
        wrist      (x, y) raw wrist position, optional (for movement re-arm)
        size       hand size in the same units, optional
        """
        cfg = self.cfg
        events = []

        # ---------------- no hand ----------------
        if label is None:
            if self._absent_since is None:
                self._absent_since = t
            self._hand_since = None
            self._window.clear()
            absent = t - self._absent_since
            if absent >= cfg.rearm_absent_seconds:
                self._blocked = None           # (a) re-arm by dropping the hand
            if (cfg.auto_space_seconds > 0 and absent >= cfg.auto_space_seconds
                    and not self._space_added_for_absence):
                self._append_space(events, kind="space_auto")
                self._space_added_for_absence = True
            return FrameResult(state="no_hand", events=events)

        # ---------------- hand visible ----------------
        self._absent_since = None
        self._space_added_for_absence = False
        if self._hand_since is None:
            self._hand_since = t

        confident = confidence >= cfg.min_confidence
        vote = label if confident else None
        self._window.append((t, vote))
        while self._window and self._window[0][0] < t - cfg.hold_seconds:
            self._window.popleft()

        # ---- re-arm rules (b) and (c) for a blocked letter ----
        if self._blocked is not None:
            if vote == self._blocked:
                self._blocked_last_seen = t   # still confidently the same letter
            last = self._blocked_last_seen if self._blocked_last_seen is not None else t
            gap = t - last                    # how long it has NOT been seen
            if gap >= cfg.rearm_other_seconds:
                self._blocked = None                       # (b) relaxed/changed
            elif (cfg.rearm_move_hand_sizes > 0 and wrist is not None and size
                  and self._commit_wrist is not None):
                dx = wrist[0] - self._commit_wrist[0]
                dy = wrist[1] - self._commit_wrist[1]
                if (dx * dx + dy * dy) ** 0.5 >= cfg.rearm_move_hand_sizes * size:
                    self._blocked = None                   # (c) slide / bounce
                    self._last_commit_t = t                # need a fresh hold

        # ---- find the candidate in the hold window ----
        votes = [v for _, v in self._window]
        counts = {}
        for v in votes:
            if v is not None:
                counts[v] = counts.get(v, 0) + 1
        candidate = max(counts, key=counts.get) if counts else None

        start = self._hand_since
        if self._last_commit_t is not None:
            start = max(start, self._last_commit_t)
        held = t - start
        share = counts.get(candidate, 0) / len(votes) if votes else 0.0
        n_frames = sum(1 for tt, _ in self._window if tt >= start)
        progress = min(1.0, held / cfg.hold_seconds) if candidate else 0.0
        blocked = candidate is not None and candidate == self._blocked

        result = FrameResult(state="holding" if confident else "uncertain",
                             label=label, confidence=float(confidence),
                             candidate=candidate, progress=progress,
                             blocked=blocked, events=events)

        if (candidate is not None and not blocked
                and held >= cfg.hold_seconds - 1e-9
                and n_frames >= cfg.min_frames
                and share >= cfg.agreement):
            self._commit(candidate, t, wrist, events)
            result.state = "accepted"
            result.accepted = candidate
            result.progress = 1.0
            result.blocked = True
        return result

    def _commit(self, label, t, wrist, events):
        if label in CONTROL_LABELS:
            if self.cfg.control_gestures:
                if label == "space":
                    self._append_space(events)
                else:
                    events.extend(self.delete())
        else:
            self._append_letter(label, events)
        self._blocked = label
        self._blocked_last_seen = t
        self._last_commit_t = t
        self._commit_wrist = tuple(wrist) if wrist is not None else None
        self._window.clear()
''',
    "asl/seq/__init__.py": r'''"""
asl.seq - EXPERIMENTAL continuous-fingerspelling backend (TCN + CTC).

This package is separate from the baseline (asl/classifier.py and
baseline/model/), which it never imports or modifies.

    camera / dataset landmarks
      -> features.SequenceFeaturizer   68 numbers per frame, causal (no future frames)
      -> features.Normalizer            mean/std fitted on the TRAINING split only
      -> model.build_tcn_ctc            causal temporal convolution network
      -> ctc.greedy_decode / StreamingGreedyDecoder   -> ordered letters A-Z

Modules
-------
vocab      blank = 0, A-Z = 1..26, transcript normalisation and filtering
features   68-feature definition, causal featurizer, normaliser, resampler
ctc        CTC decoding (offline + streaming), edit distance, metrics
model      TCN-CTC architecture, receptive field, save/load of inference bundles
data       Google ASL Fingerspelling metadata, subset selection, extraction,
           signer split, manifests, batching, augmentation
train      explicit CTC training loop, resumable checkpoints
evaluate   evaluation report (CER, word accuracy, repeats, speed, idle)
inference  offline (Start/Finish) and streaming recognisers for local use
fsboard    Google's FSboard dataset: choosing clips, downloads, decoding the
           landmark records, exporting compact training examples

Only numpy is needed to import vocab/features/ctc. TensorFlow is imported
lazily by model/train/inference; pandas/pyarrow only by data (Colab side).
"""
''',
    "asl/seq/vocab.py": r'''"""
vocab.py - Output vocabulary for the CTC model.

    index 0      CTC blank  ("no new letter at this frame"; NOT a space)
    index 1..26  A..Z
    index 27     space between words (added in Phase 2B, vocabulary v2)

The blank never appears in a target transcript. Digits and punctuation are
not in the vocabulary, so transcripts containing them are EXCLUDED (the whole
recording is skipped). We never delete characters from a transcript while
keeping the hand movements that produced them, because the model would then
learn that those movements mean "nothing".

Spaces: a phrase such as "I LOVE YOU" may be used only where the caller asks
for it (allow_spaces=True), and only with single spaces between words. Older
models have 27 outputs (no space); index 0-26 mean the same in both versions,
so those models still load and read exactly as before.
"""
import re
import string

BLANK = 0
LETTERS = string.ascii_uppercase                    # "ABC...Z"
SPACE = " "
VOCAB = ["<blank>"] + list(LETTERS) + [SPACE]       # index -> symbol
NUM_CLASSES = len(VOCAB)                            # 28 outputs per frame
LETTERS_ONLY_CLASSES = 27                           # models made before Phase 2B
CHAR_TO_ID = {c: i for i, c in enumerate(VOCAB) if i != BLANK}
SPACE_ID = CHAR_TO_ID[SPACE]

VOCAB_VERSION = "blank0-AZ-space-v2"
_WORD = re.compile(r"^[A-Z]+$")
_PHRASE = re.compile(r"^[A-Z]+( [A-Z]+)*$")         # words separated by single spaces


def normalise_transcript(text):
    """Consistent case: strip surrounding whitespace, upper-case.

    Internal characters are NOT removed; incompatible text stays incompatible.
    """
    return str(text).strip().upper()


def is_compatible(text, allow_spaces=False):
    """True if the normalised transcript uses only A-Z (one complete word), or with
    allow_spaces=True, A-Z words separated by single spaces.
    Non-ASCII text never qualifies, even if upper() would turn it into A-Z."""
    pattern = _PHRASE if allow_spaces else _WORD
    return str(text).isascii() and bool(pattern.match(normalise_transcript(text)))


def encode(text):
    """'HELLO' -> [8, 5, 12, 12, 15]; 'HI YOU' -> [8, 9, 27, 25, 15, 21].
    Raises ValueError on anything else."""
    if not is_compatible(text, allow_spaces=True):
        raise ValueError(f"Transcript not in the A-Z + space vocabulary: {text!r}")
    return [CHAR_TO_ID[c] for c in normalise_transcript(text)]


def decode_ids(ids):
    """[8, 5, 12, 12, 15] -> 'HELLO' (blanks are skipped; 27 is a space)."""
    return "".join(VOCAB[i] for i in ids if i != BLANK)


def required_frames(label_ids):
    """Minimum number of output frames CTC needs to produce this label.

    One frame per symbol, plus one blank between every pair of identical
    neighbouring letters (HELLO needs 6: H E L <blank> L O).
    """
    repeats = sum(1 for a, b in zip(label_ids, label_ids[1:]) if a == b)
    return len(label_ids) + repeats


def is_known_vocabulary(symbols):
    """True for this vocabulary or an older one it extends (the 27-output A-Z version)."""
    symbols = list(symbols)
    return len(symbols) >= LETTERS_ONLY_CLASSES and symbols == VOCAB[:len(symbols)]
''',
    "asl/seq/features.py": r'''"""
features.py - The 68 numbers the sequence model sees for every video frame.

Why not reuse the baseline's 63 numbers alone?  The baseline subtracts the
wrist, so the hand is always "at the origin". That is right for a single
held letter, but it erases the movement of the whole hand (J, Z, the small
slide of a double letter, the transition between letters). So we keep it
and add where the wrist is and how fast it moves.

Per frame (FEATURE_NAMES lists them in order):
    0..62  finger coordinates: the baseline asl.features.normalise_points()
           (wrist moved to the origin, divided by hand size, x0,y0,z0,...).
           Indices 0..2 are the wrist itself and are always 0.
    63,64  wrist position x, y in image coordinates minus 0.5 (centre = 0)
    65,66  wrist velocity x, y in HAND SIZES PER FRAME (so the number does
           not depend on how close the hand is to the camera)
    67     validity flag: 1 = a hand was tracked in this frame, 0 = not

Rules (all causal = only the current and past frames are used):
* Signing hand: one hand per sequence (the person's signing hand). A LEFT
  hand is mirrored (x -> 1 - x) so every sequence looks like a right hand.
* Missing / invalid detection (None, NaN, inf, or a hand smaller than
  min_hand_scale): all 68 numbers are 0, including the flag.
* Velocity: difference to the LAST VALID frame divided by the number of
  frames since then, if that gap is <= max_gap_frames; otherwise 0 (after a
  long gap the old position says nothing about current movement).
  The first valid frame of a sequence has velocity 0. Clipped to +-vel_clip.
* Frame rate: the model counts in frames, so live camera frames are
  resampled to target_fps (CausalResampler) before featurisation.

The SAME SequenceFeaturizer.step() is used for training data (Colab) and
the live camera (laptop), so the two cannot drift apart.
"""
import numpy as np

from ..features import normalise_points, NUM_LANDMARKS

FEATURE_VERSION = "seq68-v1"

FEATURE_NAMES = ([f"hand{i}_{a}" for i in range(NUM_LANDMARKS) for a in "xyz"]
                 + ["wrist_x", "wrist_y", "wrist_vx", "wrist_vy", "valid"])
NUM_FEATURES = len(FEATURE_NAMES)                   # 68
VALID = NUM_FEATURES - 1                            # index of the flag
WRIST_POS = slice(63, 65)
WRIST_VEL = slice(65, 67)
FINGERS = slice(0, 63)

DEFAULT_CONFIG = {
    "feature_version": FEATURE_VERSION,
    "num_features": NUM_FEATURES,
    "feature_names": FEATURE_NAMES,
    "canonical_hand": "right",       # left hands are mirrored to look like right hands
    "target_fps": 30.0,              # ASSUMPTION about the dataset; see docs/SEQUENCE_MODEL.md
    "max_gap_frames": 3,
    "vel_clip": 1.0,                 # hand sizes per frame
    "min_hand_scale": 1e-3,          # smaller "hands" are treated as tracking failures
}


def canonical_points(points, hand):
    """Mirror a left hand so that all sequences use right-hand geometry.
    points: (..., 21, 3), e.g. one frame or a whole (T, 21, 3) sequence."""
    pts = np.array(points, dtype=np.float32, copy=True)
    if str(hand).lower().startswith("l"):
        pts[..., 0] = 1.0 - pts[..., 0]
    return pts


def _hand_scale(pts):
    return float(np.max(np.linalg.norm(pts[:, :2] - pts[0, :2], axis=1)))


class SequenceFeaturizer:
    """Turns one frame at a time into 68 features, remembering only the past."""

    def __init__(self, config=None):
        self.cfg = dict(DEFAULT_CONFIG, **(config or {}))
        self.reset()

    def reset(self):
        self._last_wrist = None
        self._since_valid = 0

    def step(self, points):
        """points: (21, 3) raw landmarks of the (already canonical) signing
        hand, or None when no hand was found. Returns float32 (68,)."""
        out = np.zeros(NUM_FEATURES, np.float32)
        pts = None
        if points is not None:
            pts = np.asarray(points, dtype=np.float32).reshape(NUM_LANDMARKS, 3)
            if not np.all(np.isfinite(pts)) or _hand_scale(pts) < self.cfg["min_hand_scale"]:
                pts = None
        self._since_valid += 1
        if pts is None:
            return out

        scale = _hand_scale(pts)
        wrist = pts[0, :2]
        out[FINGERS] = normalise_points(pts)
        out[WRIST_POS] = wrist - 0.5
        gap = self._since_valid
        if self._last_wrist is not None and gap <= self.cfg["max_gap_frames"]:
            vel = (wrist - self._last_wrist) / gap / scale
            c = self.cfg["vel_clip"]
            out[WRIST_VEL] = np.clip(vel, -c, c)
        out[VALID] = 1.0
        self._last_wrist = wrist.copy()
        self._since_valid = 0
        return out

    def featurize(self, points_seq):
        """(T, 21, 3) with NaN rows for missing frames (or a list that may
        contain None) -> (T, 68). Resets first."""
        self.reset()
        rows = [self.step(p) for p in points_seq]
        return np.stack(rows) if rows else np.zeros((0, NUM_FEATURES), np.float32)


def featurize_sequence(points_seq, config=None):
    return SequenceFeaturizer(config).featurize(points_seq)


class Normalizer:
    """Per-feature mean/std, fitted on VALID frames of the TRAINING split only.

    Invalid frames stay all-zero after normalisation; the flag is never scaled.
    """

    def __init__(self, mean=None, std=None):
        self.mean = None if mean is None else np.asarray(mean, np.float32)
        self.std = None if std is None else np.asarray(std, np.float32)

    def fit(self, feature_arrays):
        n = 0
        s = np.zeros(NUM_FEATURES, np.float64)
        ss = np.zeros(NUM_FEATURES, np.float64)
        for f in feature_arrays:
            v = f[f[:, VALID] > 0.5].astype(np.float64)
            n += len(v)
            s += v.sum(0)
            ss += (v ** 2).sum(0)
        if n == 0:
            raise ValueError("No valid frames to fit the normaliser")
        mean = s / n
        std = np.sqrt(np.maximum(ss / n - mean ** 2, 0.0))
        std[std < 1e-6] = 1.0                # constant columns (wrist = origin) stay as they are
        mean[VALID], std[VALID] = 0.0, 1.0
        self.mean, self.std = mean.astype(np.float32), std.astype(np.float32)
        return self

    def apply(self, feats):
        f = np.asarray(feats, np.float32)
        valid = f[..., VALID:VALID + 1] > 0.5
        out = np.where(valid, (f - self.mean) / self.std, 0.0).astype(np.float32)
        out[..., VALID] = valid[..., 0]
        return out

    def to_dict(self):
        return {"mean": self.mean.tolist(), "std": self.std.tolist()}

    @classmethod
    def from_dict(cls, d):
        return cls(d["mean"], d["std"])


def preprocessing_dict(normalizer, feature_cfg=None, **extra):
    """Everything the laptop needs to reproduce training preprocessing."""
    return {"features": dict(DEFAULT_CONFIG, **(feature_cfg or {})),
            "normalizer": normalizer.to_dict(), **extra}


class CausalResampler:
    """Converts camera frames with timestamps to a fixed frame rate.

    For every target tick between the previous and the current camera frame
    it linearly interpolates the landmarks (both frames valid) or takes the
    nearer frame. Only frames that have already arrived are used, so the
    extra delay is at most one camera frame.
    """

    def __init__(self, target_fps=30.0):
        self.dt = 1.0 / float(target_fps)
        self.reset()

    def reset(self):
        self._prev = None            # (t, points or None)
        self._next_tick = None

    def push(self, t, points):
        """Add one camera frame; returns a list of resampled frames (points or None)."""
        pts = None if points is None else np.asarray(points, np.float32).reshape(NUM_LANDMARKS, 3)
        out = []
        if self._prev is None:
            out.append(pts)
            self._next_tick = t + self.dt
        else:
            t0, p0 = self._prev
            while self._next_tick <= t + 1e-9:
                tau = self._next_tick
                w = (tau - t0) / (t - t0) if t > t0 else 1.0
                if p0 is not None and pts is not None:
                    out.append((1 - w) * p0 + w * pts)
                else:
                    out.append(p0 if w < 0.5 else pts)
                self._next_tick += self.dt
        self._prev = (t, pts)
        return out


def chirality_score(points_seq):
    """Fraction of valid frames where the thumb base (landmark 1) is to the
    RIGHT of the little-finger base (landmark 17) in the image.

    Used only as a sanity check that the dataset's canonical hand and the
    live camera's canonical hand have the same left/right geometry (mirror
    convention). Values near 0 and near 1 mean opposite conventions.
    """
    p = np.asarray(points_seq, np.float32).reshape(-1, NUM_LANDMARKS, 3)
    ok = np.all(np.isfinite(p.reshape(len(p), -1)), axis=1)
    if not ok.any():
        return float("nan")
    return float(np.mean(p[ok, 1, 0] > p[ok, 17, 0]))
''',
    "asl/seq/ctc.py": r'''"""
ctc.py - Turning per-frame scores into letters, and measuring errors.

CTC in one paragraph: at every frame the model gives a score for "blank"
and for each letter. Greedy decoding takes the best symbol per frame, then
  1) merges neighbouring identical symbols  (L L L -> L)
  2) removes blanks.
So "L L <blank> L" -> "LL" (a real double letter) but "L L L" -> "L".
A blank between two identical letters is the ONLY way to produce a double
letter, which is why vocab.required_frames() counts one extra frame for it.

Greedy decoding only; beam search / word lists can come later.
"""
import numpy as np

from .vocab import BLANK, decode_ids


def greedy_decode(scores, length=None):
    """scores: (T, C) logits or probabilities. Returns list of label ids."""
    s = np.asarray(scores)
    if length is not None:
        s = s[:int(length)]
    best = s.argmax(-1)
    out, prev = [], BLANK
    for k in best:
        if k != BLANK and k != prev:
            out.append(int(k))
        prev = k
    return out


def greedy_decode_text(scores, length=None):
    return decode_ids(greedy_decode(scores, length))


class StreamingGreedyDecoder:
    """Greedy CTC decoding fed a few frames at a time.

    It remembers the last frame's best symbol across chunks, so a letter
    whose frames are split over two chunks is emitted once, and
    "L | <blank> L" across a boundary still gives a double letter.
    Each frame is decoded exactly once; the result equals greedy_decode()
    on the whole recording.

    confirmed   letters whose run of frames has ended (a different symbol
                followed). The causal model never revises them.
    provisional the letter currently being emitted (its run is still going
                on); it becomes confirmed as soon as the run ends.
    """

    def __init__(self):
        self.reset()

    def reset(self):
        self.confirmed = []
        self._prev = BLANK
        self._frames = 0

    @property
    def provisional(self):
        return [] if self._prev == BLANK else [int(self._prev)]

    def push(self, scores):
        """scores: (n, C) for NEW frames only. Returns letters confirmed by this call."""
        new = []
        for k in np.asarray(scores).argmax(-1):
            if k != self._prev and self._prev != BLANK:
                new.append(int(self._prev))           # previous run just ended
            self._prev = k
            self._frames += 1
        self.confirmed.extend(new)
        return new

    def finish(self):
        """End of recording: the provisional letter (if any) becomes confirmed."""
        tail = self.provisional
        self.confirmed.extend(tail)
        self._prev = BLANK
        return tail

    def text(self):
        return decode_ids(self.confirmed), decode_ids(self.provisional)


def edit_distance(a, b):
    """Levenshtein distance between two sequences (insert/delete/substitute = 1)."""
    a, b = list(a), list(b)
    prev = list(range(len(b) + 1))
    for i, x in enumerate(a, 1):
        cur = [i]
        for j, y in enumerate(b, 1):
            cur.append(min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + (x != y)))
        prev = cur
    return prev[-1]


def edit_operations(ref, hyp):
    """Counts of (substitutions, insertions, deletions) on one best alignment."""
    ref, hyp = list(ref), list(hyp)
    n, m = len(ref), len(hyp)
    d = np.zeros((n + 1, m + 1), np.int32)
    d[:, 0] = np.arange(n + 1)
    d[0, :] = np.arange(m + 1)
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            d[i, j] = min(d[i - 1, j] + 1, d[i, j - 1] + 1, d[i - 1, j - 1] + (ref[i - 1] != hyp[j - 1]))
    sub = ins = dele = 0
    i, j = n, m
    while i > 0 or j > 0:
        if i > 0 and j > 0 and d[i, j] == d[i - 1, j - 1] + (ref[i - 1] != hyp[j - 1]):
            sub += ref[i - 1] != hyp[j - 1]
            i, j = i - 1, j - 1
        elif j > 0 and d[i, j] == d[i, j - 1] + 1:
            ins, j = ins + 1, j - 1
        else:
            dele, i = dele + 1, i - 1
    return int(sub), int(ins), int(dele)


def character_error_rate(refs, hyps):
    """Total edits / total reference letters (corpus CER, not an average of ratios)."""
    edits = sum(edit_distance(r, h) for r, h in zip(refs, hyps))
    chars = sum(len(r) for r in refs)
    return edits / max(chars, 1)


def has_double_letter(word):
    return any(a == b for a, b in zip(word, word[1:]))


def collapse_doubles(word):
    out = []
    for c in word:
        if not out or out[-1] != c:
            out.append(c)
    return "".join(out)
''',
    "asl/seq/model.py": r'''"""
model.py - The compact causal TCN with a CTC output, and its saved bundle.

TCN = temporal convolution network: small filters slide along TIME. Each
filter looks at the current frame and a few PAST frames ("causal": never
future ones), so the model can run live without waiting for the end.
Dilation spaces those few frames out (1, 2, 4, 8 frames apart), so four
blocks together see 61 frames (about 2 s at 30 fps) at a low cost.

    features (batch, T, 68)
      -> Dense 64 per frame (projection) + LayerNorm + ReLU
      -> 4 residual blocks, dilation 1, 2, 4, 8:
           [causal Conv1D(64, k=3) -> LayerNorm -> ReLU -> Dropout] x 2  + skip
      -> Dense 27 per frame = raw scores (logits): blank + A..Z
    output (batch, T, 27)   - one output per input frame (no downsampling)

LayerNormalization normalises the 64 channels of each frame separately, so
it never mixes time steps (keeps the model causal and padding-safe).
The settings below are a first experiment, not tuned optima.
"""
import json
from pathlib import Path

import numpy as np

from .features import NUM_FEATURES
from .vocab import NUM_CLASSES, SPACE_ID, is_known_vocabulary

DEFAULT_MODEL_CONFIG = {
    "architecture": "causal-tcn-ctc-v1",
    "num_features": NUM_FEATURES,
    "num_classes": NUM_CLASSES,
    "channels": 64,
    "kernel_size": 3,
    "dilations": [1, 2, 4, 8],
    "convs_per_block": 2,
    "dropout": 0.1,
}


def receptive_field(cfg=None):
    """Number of input frames (current + past) that one output can see."""
    c = dict(DEFAULT_MODEL_CONFIG, **(cfg or {}))
    return 1 + c["convs_per_block"] * (c["kernel_size"] - 1) * sum(c["dilations"])


def build_tcn_ctc(cfg=None):
    from tensorflow import keras
    from tensorflow.keras import layers

    c = dict(DEFAULT_MODEL_CONFIG, **(cfg or {}))
    inp = keras.Input(shape=(None, c["num_features"]), name="features")
    x = layers.Dense(c["channels"], name="proj")(inp)
    x = layers.LayerNormalization(name="proj_norm")(x)
    x = layers.ReLU(name="proj_relu")(x)
    for b, d in enumerate(c["dilations"]):
        h = x
        for j in range(c["convs_per_block"]):
            h = layers.Conv1D(c["channels"], c["kernel_size"], dilation_rate=d,
                              padding="causal", name=f"b{b}_conv{j}")(h)
            h = layers.LayerNormalization(name=f"b{b}_norm{j}")(h)
            h = layers.ReLU(name=f"b{b}_relu{j}")(h)
            h = layers.Dropout(c["dropout"], name=f"b{b}_drop{j}")(h)
        x = layers.Add(name=f"b{b}_residual")([x, h])
    logits = layers.Dense(c["num_classes"], name="logits")(x)
    return keras.Model(inp, logits, name="tcn_ctc")


# ------------------------------------------------------------- bundle I/O
# An inference bundle is a small folder that the laptop needs:
#   model_config.json      architecture settings (rebuilt with build_tcn_ctc)
#   model.weights.h5       trained weights only (robust across Keras 3 versions)
#   preprocessing.json     feature settings + normaliser mean/std
#   vocab.json             index -> symbol
#   reference_io.npz       a few inputs and the logits Colab produced for them,
#                          to check the laptop reproduces the same numbers
#   versions.json          package versions used for training
#   evaluation_report.json (added by evaluate.py)

def save_inference_bundle(folder, model, model_cfg, preprocessing, versions=None,
                          reference_inputs=None):
    from .vocab import VOCAB, VOCAB_VERSION

    folder = Path(folder)
    folder.mkdir(parents=True, exist_ok=True)
    model.save_weights(str(folder / "model.weights.h5"))
    cfg = dict(DEFAULT_MODEL_CONFIG, **model_cfg)
    (folder / "model_config.json").write_text(json.dumps(cfg, indent=2))
    (folder / "preprocessing.json").write_text(json.dumps(preprocessing, indent=2))
    n = int(cfg["num_classes"])                   # a 27-output model keeps the A-Z vocabulary
    (folder / "vocab.json").write_text(json.dumps({
        "version": VOCAB_VERSION if n == len(VOCAB) else "blank0-AZ-v1",
        "symbols": VOCAB[:n], "blank_index": 0}, indent=2))
    if versions:
        (folder / "versions.json").write_text(json.dumps(versions, indent=2))
    if reference_inputs is not None:
        xs = [np.asarray(x, np.float32) for x in reference_inputs]
        logits = [model(x[None], training=False).numpy()[0] for x in xs]
        np.savez_compressed(folder / "reference_io.npz",
                            **{f"x{i}": x for i, x in enumerate(xs)},
                            **{f"y{i}": y for i, y in enumerate(logits)})
    return folder


def load_inference_bundle(folder):
    """Returns (model, model_cfg, preprocessing dict)."""
    folder = Path(folder)
    model_cfg = json.loads((folder / "model_config.json").read_text())
    prep = json.loads((folder / "preprocessing.json").read_text())
    vocab = json.loads((folder / "vocab.json").read_text())
    if (vocab.get("blank_index") != 0 or len(vocab["symbols"]) != model_cfg["num_classes"]
            or not is_known_vocabulary(vocab["symbols"])):
        raise ValueError("vocab.json does not match this code's vocabulary (blank=0, A-Z, optional space)")
    model = build_tcn_ctc(model_cfg)
    model.load_weights(str(folder / "model.weights.h5"))
    return model, model_cfg, prep


def has_space_output(model_cfg):
    """Models made before Phase 2B have 27 outputs and can never write a space."""
    return int(model_cfg["num_classes"]) > SPACE_ID


def warm_start(model, source_bundle):
    """Copy a trained bundle's weights into a new model with the same layers.

    Every layer must match, except the output layer may have MORE outputs (27 -> 28,
    adding the space): its existing outputs are copied and the new ones keep their
    fresh starting values. So the new model starts out reading letters as well as
    the source model did, and only has to learn the space.
    Returns {"copied": [...], "extended": [...]} layer names. Raises ValueError if the
    layers differ (for example different dilations)."""
    source, source_cfg, _ = load_inference_bundle(source_bundle)
    by_name = {layer.name: layer for layer in source.layers}
    copied, extended = [], []
    for layer in model.layers:
        new = layer.get_weights()
        if not new:
            continue
        if layer.name not in by_name:
            raise ValueError(f"The source model has no layer {layer.name!r}: the two models are built "
                             f"differently (source dilations {source_cfg['dilations']}).")
        old = by_name[layer.name].get_weights()
        if [w.shape for w in old] == [w.shape for w in new]:
            layer.set_weights(old)
            copied.append(layer.name)
        elif (layer.name == "logits" and len(old) == len(new)
              and all(o.shape[:-1] == n.shape[:-1] and o.shape[-1] <= n.shape[-1] for o, n in zip(old, new))):
            for o, n in zip(old, new):
                n[..., :o.shape[-1]] = o
            layer.set_weights(new)
            extended.append(layer.name)
        else:
            raise ValueError(f"Layer {layer.name!r} has different sizes in the two models "
                             f"({[w.shape for w in old]} vs {[w.shape for w in new]}).")
    return {"copied": copied, "extended": extended}


def check_reference_io(folder, model, atol=1e-3):
    """Max |difference| between saved reference logits and this machine's logits."""
    path = Path(folder) / "reference_io.npz"
    if not path.exists():
        return None
    z = np.load(path)
    worst = 0.0
    i = 0
    while f"x{i}" in z:
        y = model(z[f"x{i}"][None], training=False).numpy()[0]
        worst = max(worst, float(np.max(np.abs(y - z[f"y{i}"]))))
        i += 1
    return {"max_abs_diff": worst, "ok": worst <= atol, "n": i}
''',
    "asl/seq/fsboard.py": r'''"""
fsboard.py - Google's FSboard dataset (Kaggle: googleai/fsboard).

Chooses recordings from the metadata, downloads single files, and turns
landmark records into compact training examples. Used by
notebooks/fsboard_prepare_data_colab.ipynb.

What the real files look like (checked on real data, see docs/SEQUENCE_MODEL.md)

Metadata   daun_v3/metadata/daun_v3-train.json: a JSON list, one object per
           clip, with clipFilename, phrase, signerId, clipFileMd5, ...
Videos     daun_v3/video_clips/daun_v3-train/<clipFilename>
Landmarks  daun_v3/landmarks/daun_v3-train.arrow-NNNNN-of-00100: an Arrow file
           with one "serialized" column; each row is a tf.train.SequenceExample
           whose context holds video_name, prompt, signer_id, num_frames, ...

Two surprises the code handles:

1. Names differ. A record says  ..._619.011_625.644.mp4  (seconds) where the
   metadata says  ..._619011_625644.mp4  (milliseconds). They are matched with
   an explicit rule, then the phrase and signer are compared as well. Records
   are never paired by row position.

2. Hand coordinates are not what the names suggest. x_right_hand, y_right_hand
   and z_right_hand each hold 21 numbers, but they are three overlapping
   windows of one flattened list:
       x list = x0 y0 z0 x1 y1 z1 ... x6 y6 z6
       y list =    y0 z0 x1 y1 z1 ... x6 y6 z6 x7
       z list =       z0 x1 y1 z1 ... x6 y6 z6 x7 y7
   So only 23 different numbers exist per hand per frame: landmarks 0-6
   complete, and x and y of landmark 7. The index fingertip and the middle,
   ring and little fingers are not stored. Layout is detected from the numbers
   of every frame, so a corrected dataset (21 true x, y and z values) would be
   read as complete hands without changing this code.
"""
import base64
import csv
import hashlib
import html
import json
import os
import re
import shutil
import time
import urllib.error
import urllib.parse
import urllib.request
from collections import Counter
from decimal import Decimal
from pathlib import Path

import numpy as np

from . import vocab

BASE_URL = "https://www.kaggle.com/api/v1/datasets/download/googleai/fsboard/"
DATASET_PAGE = "https://www.kaggle.com/datasets/googleai/fsboard"
TRAIN_METADATA = "daun_v3/metadata/daun_v3-train.json"
TRAIN_VIDEO_DIR = "daun_v3/video_clips/daun_v3-train/"
NUM_TRAIN_SHARDS = 100
FORMAT_VERSION = "fsboard-npz-v1"

REQUIRED_FIELDS = ("signingType", "clipFilename", "phrase", "signerId", "clipFileMd5",
                   "clipFileSize", "sensitiveContent")
MIN_LETTERS, MAX_LETTERS = 2, 12

# MediaPipe's hand landmark numbering. asl.features.points_from_mediapipe()
# returns the same order, so webcam code can index it with landmark_ids.
HAND_LANDMARK_NAMES = (
    "wrist", "thumb_cmc", "thumb_mcp", "thumb_ip", "thumb_tip",
    "index_mcp", "index_pip", "index_dip", "index_tip",
    "middle_mcp", "middle_pip", "middle_dip", "middle_tip",
    "ring_mcp", "ring_pip", "ring_dip", "ring_tip",
    "pinky_mcp", "pinky_pip", "pinky_dip", "pinky_tip")
HAND_BONES = ((0, 1), (1, 2), (2, 3), (3, 4), (0, 5), (5, 6), (6, 7), (7, 8),
              (0, 9), (9, 10), (10, 11), (11, 12), (0, 13), (13, 14), (14, 15), (15, 16),
              (0, 17), (17, 18), (18, 19), (19, 20))

LAYOUT_PARTIAL = "partial-8-landmarks"     # the shifted windows described above
LAYOUT_COMPLETE = "complete-21-landmarks"  # 21 true x, y and z values

FORMAT_NOTES = """\
FSboard pilot sequences: one .npz file per recording, plus index.csv.
Load with numpy.load(path, allow_pickle=False). No pickled objects are used.

left_hand, right_hand   float32, shape (frames, landmarks, 3). NaN where not known.
                        Read as hand[frame, position, coordinate].
                        coordinate: 0 = x, 1 = y, 2 = z.
                        position: the n-th entry of landmark_ids.
landmark_ids            MediaPipe hand landmark number of each position
                        (0 wrist, 1-4 thumb, 5-8 index, 9-12 middle, 13-16 ring,
                        17-20 little finger).
left_mask, right_mask   bool, shape (frames,). True = that hand was detected.
timestamp_us            int64, time of each frame in microseconds, as stored.
frame_index             int32, 0, 1, 2, ... in the original frame order.
answer, label_ids       the full written answer in capitals, and the same letters
                        as numbers (A=1 ... Z=26; 0 is the CTC blank, never in a label).
phrase                  the original text from the metadata.
clip_id, video_name     the clip's metadata filename, and its name in the landmark record.
signer_id, split        the signer, and FSboard's official split (always train here).
image_width, image_height, frame_rate   of the source video.
source_shard, source_record   where it was read from. Tracing only, never used for pairing.
layout                  partial-8-landmarks or complete-21-landmarks.

Coordinates: x is a fraction of the image width from the left, y a fraction of
the height from the top, z a relative depth from MediaPipe. Values can fall
slightly outside 0..1. They refer to the upright video (image_width by
image_height, usually portrait). Left and right are the labels stored in the
dataset; the tracker sometimes gives one physical hand both labels.

Important: with layout partial-8-landmarks each hand has only the wrist, the
four thumb joints and three index-finger joints, and z of landmark 7 is always
NaN. The index fingertip and the middle, ring and little fingers are not in
FSboard's landmark files. To give the same input, webcam code must take
landmarks 0-7 from MediaPipe's 21 and ignore the z of landmark 7.
"""


PARTIAL_WARNING = """\
LIMITATION: these sequences hold 8 of the 21 hand points in each frame: the wrist,
the four thumb joints and three index-finger joints. FSboard's landmark files do not
contain the index fingertip or the middle, ring and little fingers. The files are
correctly paired and readable, but many letters differ mainly by those missing fingers."""


class FsboardError(RuntimeError):
    """A problem with a plain-English explanation of what to check."""

    def _render_traceback_(self):              # notebooks show the message, not a stack trace
        return [f"STOPPED: {self}"]


# ================================================================= downloads
def kaggle_auth_from_colab_secrets():
    """Copy Kaggle credentials from Colab's Secrets panel into this process's
    environment. Nothing is printed or written to disk. Returns the names found."""
    from google.colab import userdata  # only exists inside Colab

    found = []
    for name in ("KAGGLE_API_TOKEN", "KAGGLE_USERNAME", "KAGGLE_KEY"):
        try:
            value = userdata.get(name)
        except Exception:
            value = None
        if value:
            os.environ[name] = value
            found.append(name)
    return found


class _NoRedirect(urllib.request.HTTPRedirectHandler):
    def redirect_request(self, *args, **kwargs):
        return None


def _auth_headers():
    """Optional Kaggle credentials from the environment. The public files
    downloaded without any when this was written."""
    token = os.environ.get("KAGGLE_API_TOKEN")
    if token:
        return {"Authorization": f"Bearer {token}"}
    user, key = os.environ.get("KAGGLE_USERNAME"), os.environ.get("KAGGLE_KEY")
    if user and key:
        raw = base64.b64encode(f"{user}:{key}".encode()).decode()
        return {"Authorization": f"Basic {raw}"}
    return {}


def _http_problem(code, path):
    if code in (401, 403):
        return FsboardError(
            f"Kaggle refused to give us {path} (HTTP {code}).\nWhat to check:\n"
            f"  1. Sign in at {DATASET_PAGE} and accept anything the page asks you to accept.\n"
            "  2. Kaggle > Settings > API > Create New Token. In Colab, open the key icon (Secrets),\n"
            "     add it as KAGGLE_API_TOKEN (or add KAGGLE_USERNAME and KAGGLE_KEY), and switch on\n"
            "     notebook access.\n"
            "  3. Run the Setup cell again, then this cell. Never paste a key into a code cell.")
    if code == 404:
        return FsboardError(
            f"Kaggle says this file does not exist: {path} (HTTP 404).\nWhat to check:\n"
            f"  1. Look for it on the Data tab of {DATASET_PAGE}\n"
            "  2. If the folders were renamed, update the path in the Setup code.")
    if code == 429:
        return FsboardError(f"Kaggle asked us to slow down while fetching {path} (HTTP 429).\n"
                            "What to check: wait a few minutes, then run this cell again.")
    return FsboardError(f"The download of {path} failed with HTTP {code}.\n"
                        "What to check: run the cell again in a minute. If it keeps failing, "
                        f"see whether {DATASET_PAGE} opens in your browser.")


def _resolve(path):
    """Ask Kaggle where a file lives. Returns (url, headers) for fetching it.

    Kaggle answers with a redirect to a time-limited storage link. Credentials,
    if any, go to Kaggle only and are not forwarded to the storage host.
    """
    url = BASE_URL + urllib.parse.quote(path, safe="")
    auth = _auth_headers()
    req = urllib.request.Request(url, headers={"User-Agent": "fsboard-notebook", **auth})
    try:
        with urllib.request.build_opener(_NoRedirect).open(req, timeout=60):
            return url, auth                   # served directly, no redirect
    except urllib.error.HTTPError as e:
        if e.code in (301, 302, 303, 307, 308) and e.headers.get("Location"):
            return e.headers["Location"], {}
        raise _http_problem(e.code, path) from None
    except (urllib.error.URLError, TimeoutError, OSError) as e:
        raise FsboardError(f"Could not reach Kaggle to fetch {path} ({e}).\n"
                           "What to check: is this Colab session still connected? "
                           "Run the cell again.") from None


def remote_size(path):
    """Size in bytes, read from the server without downloading the file."""
    url, headers = _resolve(path)
    req = urllib.request.Request(url, headers={"Range": "bytes=0-0", "User-Agent": "fsboard-notebook",
                                               **headers})
    try:
        with urllib.request.urlopen(req, timeout=60) as r:
            content_range = r.headers.get("Content-Range", "")
            return int(content_range.split("/")[-1]) if "/" in content_range \
                else int(r.headers.get("Content-Length", 0))
    except urllib.error.HTTPError as e:
        raise _http_problem(e.code, path) from None
    except (urllib.error.URLError, TimeoutError, OSError) as e:
        raise FsboardError(f"Could not reach the storage server for {path} ({e}).\n"
                           "What to check: is this Colab session still connected? "
                           "Run the cell again.") from None


def file_md5(path):
    digest = hashlib.md5(usedforsecurity=False)
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            digest.update(block)
    return digest.hexdigest()


def free_gb(folder):
    folder = Path(folder)
    folder.mkdir(parents=True, exist_ok=True)
    return shutil.disk_usage(folder).free / 1e9


def _fetch_to_part(url, headers, part, expected_size, log, label):
    """One download attempt, continuing from however much part already holds."""
    have = part.stat().st_size if part.exists() else 0
    req_headers = {"User-Agent": "fsboard-notebook", **headers}
    if have:
        req_headers["Range"] = f"bytes={have}-"
    req = urllib.request.Request(url, headers=req_headers)
    with urllib.request.urlopen(req, timeout=120) as r:
        if have and r.status != 206:           # server ignored the Range: start again
            have = 0
        length = r.headers.get("Content-Length")
        total = have + int(length) if length else None
        step = max((expected_size or 0) // 10, 1)
        next_report = (have // step + 1) * step
        with open(part, "ab" if have else "wb") as out:
            while True:
                block = r.read(1 << 20)
                if not block:
                    break
                out.write(block)
                have += len(block)
                if expected_size and have >= next_report and have < expected_size:
                    log(f"   {label}: {have / 1e6:,.0f} of {expected_size / 1e6:,.0f} MB")
                    next_report += step
    if total is not None and have < total:     # a dropped connection can look like a normal end
        raise ConnectionError(f"received {have:,} of {total:,} bytes")


def download(path, dest, expected_size=None, expected_md5=None, log=print, attempts=5):
    """Download ONE dataset file to dest and verify it. Returns dest.

    A verified copy is reused, an interrupted download continues where it
    stopped, and a file that fails verification is deleted.
    """
    dest = Path(dest)
    dest.parent.mkdir(parents=True, exist_ok=True)
    label = dest.name

    def problem():
        size = dest.stat().st_size
        if expected_size is not None and size != expected_size:
            return f"it has {size:,} bytes but should have {expected_size:,}"
        if expected_md5:
            actual = file_md5(dest)
            if actual != expected_md5:
                return f"its MD5 is {actual} but clipFileMd5 says {expected_md5}"
        return ""

    if dest.exists() and not problem():
        log(f"   {label}: already here and verified, not downloading again")
        return dest
    if dest.exists():
        dest.unlink()

    url, headers = _resolve(path)
    part = dest.with_name(dest.name + ".part")
    for attempt in range(1, attempts + 1):
        try:
            _fetch_to_part(url, headers, part, expected_size, log, label)
            break
        except urllib.error.HTTPError as e:
            if e.code == 416:                  # the partial file is already complete
                break
            raise _http_problem(e.code, path) from None
        except OSError as e:                   # URLError and timeouts are OSErrors
            if getattr(e, "errno", None) == 28:
                raise FsboardError(f"The Colab disk is full while saving {label}.\nWhat to check: "
                                   f"{free_gb(dest.parent):.1f} GB is free. Delete files in the "
                                   "Files panel, or use Runtime > Disconnect and delete runtime.") from None
            if attempt == attempts:
                raise FsboardError(f"The download of {path} kept stopping ({e}).\nWhat to check: "
                                   "is this Colab session still connected? Run the cell again; "
                                   "it continues from what it already has.") from None
            log(f"   {label}: connection dropped, trying again ({attempt}/{attempts})")
            time.sleep(2 * attempt)
    part.replace(dest)
    bad = problem()
    if bad:
        dest.unlink()
        raise FsboardError(f"The downloaded {label} did not pass verification: {bad}.\n"
                           "What to check: the bad copy was deleted, so run this cell again. "
                           "If it fails twice, the file on Kaggle may have changed.")
    return dest


# ================================================================== metadata
def load_metadata(path):
    """Read a metadata JSON file and confirm the fields this code relies on."""
    with open(path, encoding="utf-8") as f:
        records = json.load(f)
    if not isinstance(records, list) or not records or not isinstance(records[0], dict):
        raise FsboardError(f"{path} is not a list of clip records.\nWhat to check: open the "
                           "file's preview on Kaggle; the dataset's format may have changed.")
    missing = [k for k in REQUIRED_FIELDS if k not in records[0]]
    if missing:
        raise FsboardError(f"The metadata has no field called {', '.join(missing)}.\nFields it does "
                           f"have: {', '.join(records[0])}.\nWhat to check: the dataset's format "
                           "may have changed; update REQUIRED_FIELDS and the code that uses them.")
    return records


def describe_fields(records):
    """(field, how many clips have it, type, example) for each metadata field."""
    counts, kinds, examples = Counter(), {}, {}
    for r in records:
        for k, v in r.items():
            counts[k] += 1
            kinds.setdefault(k, type(v).__name__)
            examples.setdefault(k, v)
    return [(k, counts[k], kinds[k], examples[k]) for k in counts]


def phrase_problem(phrase, min_letters=MIN_LETTERS, max_letters=MAX_LETTERS, allow_spaces=False):
    """Why this complete phrase cannot be a training answer ("" if it can).

    The phrase is judged as it is written. Nothing is removed from it, because
    the hands in the video spell every character. With allow_spaces=True, words
    separated by single spaces are accepted too (the space is an output since
    Phase 2B); the letter limits then count letters only.
    """
    if not isinstance(phrase, str) or not phrase:
        return "empty phrase"
    if re.fullmatch(r"[A-Za-z]+( [A-Za-z]+)*" if allow_spaces else r"[A-Za-z]+", phrase):
        letters = sum(c != " " for c in phrase)
        if letters < min_letters:
            return "too few letters"
        if letters > max_letters:
            return "too many letters"
        return ""
    if any(c.isdigit() for c in phrase):
        return "contains a digit"
    if allow_spaces and re.fullmatch(r"[A-Za-z ]+", phrase):
        return "space at the start or end, or two spaces"
    if not allow_spaces and any(c.isspace() for c in phrase):
        return "contains a space"
    return "contains a symbol"


def training_answer(phrase):
    """'letter' -> 'LETTER'. Repeated letters stay; they are real signs."""
    return vocab.normalise_transcript(phrase)


def eligible_rows(records, min_letters=MIN_LETTERS, max_letters=MAX_LETTERS, skip_sensitive=True,
                  allow_spaces=False):
    """All usable clips as simple rows, plus a count of why others were skipped.

    A clip with unsupported characters is skipped whole. Clips the publisher
    tagged sensitiveContent are left out of this pilot unless asked for.
    """
    rows, skipped = [], Counter()
    for r in records:
        reason = phrase_problem(r["phrase"], min_letters, max_letters, allow_spaces)
        if not reason and r["signingType"] != "fs":
            reason = "not fingerspelling"
        if not reason and skip_sensitive and r["sensitiveContent"]:
            reason = "tagged sensitiveContent"
        if reason:
            skipped[reason] += 1
            continue
        rows.append({"filename": r["clipFilename"], "phrase": r["phrase"],
                     "answer": training_answer(r["phrase"]), "signer_id": r["signerId"],
                     "md5": r["clipFileMd5"], "size_bytes": r["clipFileSize"]})
    return rows, skipped


def select_preview(rows, n=5):
    """Up to n clips from different signers (same result on every run)."""
    chosen, signers = [], set()
    for row in sorted(rows, key=lambda r: r["filename"]):
        if row["signer_id"] not in signers:
            chosen.append(row)
            signers.add(row["signer_id"])
        if len(chosen) == n:
            break
    if not chosen:
        raise FsboardError(f"No recording passed the filter ({MIN_LETTERS}-{MAX_LETTERS} letters and "
                           "nothing else).\nWhat to check:\n  1. Did the field list above show "
                           "'phrase' and 'signingType'?\n  2. Was the metadata download complete? "
                           "Delete it in the Files panel and run this cell again.\n  3. Are the "
                           "letter limits in this cell set the way you want?")
    return chosen


def clip_path(row):
    return TRAIN_VIDEO_DIR + row["filename"]


def pick_row(chosen, number):
    if not isinstance(number, int) or not 0 <= number < len(chosen):
        raise FsboardError(f"PREVIEW_ROW is {number!r}, but the table has rows 0 to {len(chosen) - 1}.\n"
                           "What to check: set PREVIEW_ROW to a row number from the table in step 1.")
    return chosen[number]


def video_html(path, answer, width=320):
    """The written answer directly above a playable video. Embedding is needed
    because a Colab browser tab cannot read files on the Colab disk."""
    data = base64.b64encode(Path(path).read_bytes()).decode()
    return (f'<div style="font-family:sans-serif">'
            f'<div style="font-size:14px;color:#555">Written answer</div>'
            f'<div style="font-size:30px;font-weight:700;letter-spacing:2px;margin-bottom:8px">'
            f'{html.escape(answer)}</div>'
            f'<video controls loop playsinline width="{int(width)}" '
            f'src="data:video/mp4;base64,{data}"></video></div>')


# ================================================== records in landmark shards
def train_shard_path(number):
    if not isinstance(number, int) or not 0 <= number < NUM_TRAIN_SHARDS:
        raise FsboardError(f"SHARD_NUMBER is {number!r}, but training shards are numbered 0 to "
                           f"{NUM_TRAIN_SHARDS - 1}.\nWhat to check: use a whole number in that range.")
    return f"daun_v3/landmarks/daun_v3-train.arrow-{number:05d}-of-{NUM_TRAIN_SHARDS:05d}"


def shard_split(shard_name):
    """The official split, read from the file name: daun_v3-train.arrow-... -> train."""
    m = re.search(r"-(train|validation|test)\.arrow-", str(shard_name).replace("\\", "/"))
    if not m:
        raise FsboardError(f"{shard_name} is not named like an FSboard landmark shard.")
    return m.group(1)


def open_shard(path):
    """Open an Arrow shard without reading it all. Each row is decompressed
    only when it is asked for, which is how we read just what the pilot needs."""
    import pyarrow as pa

    try:
        reader = pa.ipc.open_file(pa.memory_map(str(path), "r"))
    except (pa.ArrowInvalid, OSError) as e:
        raise FsboardError(f"{path} could not be read as an Arrow file ({e}).\nWhat to check: the "
                           "download may be incomplete. Delete it in the Files panel and run this "
                           "cell again.") from None
    if "serialized" not in reader.schema.names:
        raise FsboardError(f"{path} has columns {reader.schema.names}, not 'serialized'.\nWhat to "
                           "check: compare with convert_arrow_to_parquet.py on the dataset page.")
    return reader


def iter_serialized(reader):
    """Yield the bytes of each record, in file order."""
    for i in range(reader.num_record_batches):
        batch = reader.get_batch(i)
        for value in batch.column(batch.schema.get_field_index("serialized")):
            yield value.as_py()


def parse_example(serialized):
    try:
        import tensorflow as tf
    except ImportError:
        raise FsboardError("Reading landmark records needs TensorFlow, which this runtime does not "
                           "have.\nWhat to check: run  !pip install tensorflow  in a new cell, then "
                           "run this cell again.") from None
    example = tf.train.SequenceExample()
    example.ParseFromString(serialized)
    return example


def record_context(example):
    """The clip-level values stored inside the record itself."""
    out = {}
    for key, feature in example.context.feature.items():
        kind = feature.WhichOneof("kind")
        if not kind or not getattr(feature, kind).value:
            continue
        value = getattr(feature, kind).value[0]
        out[key] = value.decode("utf-8") if isinstance(value, bytes) else value
    return out


def clip_filename_from_video_name(video_name):
    """Explicit name rule: seconds with a decimal point -> whole milliseconds.
    e4fa_965_20220920_145746_165.367_167.833.mp4 -> ..._165367_167833.mp4"""
    m = re.fullmatch(r"(.+)_(\d+\.\d+)_(\d+\.\d+)\.mp4", video_name)
    if not m:
        return video_name
    start, end = (int(Decimal(t) * 1000) for t in m.groups()[1:])
    return f"{m.group(1)}_{start}_{end}.mp4"


def metadata_index(records):
    return {r["clipFilename"]: r for r in records}


def match_metadata(context, index):
    """Find this record's metadata row by name, then compare its phrase and
    signer with the record's own. Returns (row, "") or (None, reason)."""
    row = index.get(clip_filename_from_video_name(context.get("video_name", "")))
    if row is None:
        return None, "no metadata row with this clip name"
    if row["phrase"] != context.get("prompt"):
        return None, "phrase differs between record and metadata"
    if row["signerId"] != context.get("signer_id"):
        return None, "signer differs between record and metadata"
    return row, ""


def decode_hand_frame(x, y, z):
    """Three stored lists for one hand in one frame -> (layout, points).

    See the top of this file. points has shape (8, 3) for the partial layout
    and (21, 3) for a complete one; (None, None) means an unknown layout.
    """
    x, y, z = (np.asarray(v, np.float32) for v in (x, y, z))
    if not len(x) == len(y) == len(z) == len(HAND_LANDMARK_NAMES):
        return None, None
    if np.allclose(x[1:], y[:-1], atol=1e-7) and np.allclose(y[1:], z[:-1], atol=1e-7):
        points = np.full((8, 3), np.nan, np.float32)
        points.flat[:23] = np.concatenate([x, y[-1:], z[-1:]])
        return LAYOUT_PARTIAL, points
    return LAYOUT_COMPLETE, np.stack([x, y, z], axis=1)


def _float_values(feature):
    return feature.float_list.value if feature.WhichOneof("kind") == "float_list" else ()


def decode_hands(example):
    """Ordered hand coordinates, masks and timing from one record. A frame's
    position in the lists is its frame number, as in the publisher's script."""
    lists = example.feature_lists.feature_list
    stamps = lists["timestamp"].feature
    T = len(stamps)
    decoded, masks, layouts = {}, {}, Counter()
    for hand in ("left", "right"):
        name = f"{hand}_hand"
        xs, ys, zs = (lists[f"{a}_{name}"].feature for a in "xyz")
        if not len(xs) == len(ys) == len(zs) == T:
            raise ValueError(f"{name} lists do not have {T} frames")
        frames = {}
        for t in range(T):
            x, y, z = _float_values(xs[t]), _float_values(ys[t]), _float_values(zs[t])
            if not (x and y and z):
                continue                       # hand not detected in this frame
            layout, points = decode_hand_frame(x, y, z)
            layouts[layout] += 1
            if layout is not None:
                frames[t] = points
        decoded[hand], masks[hand] = frames, np.array([t in frames for t in range(T)], bool)
    if None in layouts or len(layouts) > 1:
        raise ValueError(f"unexpected coordinate layout {dict(layouts)}")
    layout = next(iter(layouts), LAYOUT_PARTIAL)
    count = 8 if layout == LAYOUT_PARTIAL else len(HAND_LANDMARK_NAMES)
    out = {"layout": layout, "landmark_ids": np.arange(count, dtype=np.int32),
           "timestamp_us": np.array([s.int64_list.value[0] if s.WhichOneof("kind") == "int64_list" else -1
                                     for s in stamps], np.int64),
           "frame_index": np.arange(T, dtype=np.int32)}
    for hand in ("left", "right"):
        arr = np.full((T, count, 3), np.nan, np.float32)
        for t, points in decoded[hand].items():
            arr[t] = points
        out[f"{hand}_hand"], out[f"{hand}_mask"] = arr, masks[hand]
    return out


def format_preview(example):
    """A plain-text look inside one record, so the format is shown, not assumed."""
    context = record_context(example)
    lists = example.feature_lists.feature_list
    lines = ["Clip-level values stored inside the record:"]
    lines += [f"   {k:18s} {context[k]}" for k in sorted(context)]
    lines += ["Frame-by-frame lists: " + ", ".join(sorted(lists))]
    for hand in ("right_hand", "left_hand"):
        xs, ys, zs = (lists[f"{a}_{hand}"].feature for a in "xyz")
        found = [t for t in range(len(xs)) if _float_values(xs[t])]
        lines.append(f"{hand}: detected in {len(found)} of {len(xs)} frames")
        if found:
            t = found[0]
            x, y, z = (np.array(_float_values(v[t])) for v in (xs, ys, zs))
            fmt = lambda v: " ".join(f"{a:6.3f}" for a in v[:9])
            layout, _ = decode_hand_frame(x, y, z)
            lines += [f"   frame {t}, first 9 of {len(x)} numbers in each list:",
                      f"   x list: {fmt(x)}", f"   y list: {fmt(y)}", f"   z list: {fmt(z)}",
                      f"   layout found from these numbers: {layout}"]
    return "\n".join(lines)


def build_example(context, row, hands, shard_name, record_number):
    """Everything saved for one recording. Pairing was settled by match_metadata."""
    answer = training_answer(row["phrase"])
    return dict(hands,
                clip_id=row["clipFilename"], video_name=context["video_name"],
                signer_id=row["signerId"], split=shard_split(shard_name),
                phrase=row["phrase"], answer=answer,
                label_ids=np.array(vocab.encode(answer), np.int32),
                frame_rate=float(context.get("image/frame_rate", float("nan"))),
                image_width=int(context.get("image/width", -1)),
                image_height=int(context.get("image/height", -1)),
                source_shard=Path(str(shard_name)).name, source_record=int(record_number),
                format_version=FORMAT_VERSION)


def hand_frame_count(example):
    return int(np.sum(example["left_mask"] | example["right_mask"]))


def save_example(example, folder):
    folder = Path(folder)
    folder.mkdir(parents=True, exist_ok=True)
    path = folder / (Path(example["clip_id"]).stem + ".npz")
    np.savez_compressed(path, **example)
    return path


def load_example(path):
    """Read an exported file back into plain values (no pickled objects)."""
    with np.load(path, allow_pickle=False) as z:
        out = {k: z[k] for k in z.files}
    for k, v in out.items():
        if v.ndim == 0:
            out[k] = v.item()
    return out


def check_example(example):
    """Checks on one exported recording: a list of (what, passed, detail)."""
    left, right = example["left_hand"], example["right_hand"]
    lmask, rmask = example["left_mask"], example["right_mask"]
    stamps, T = example["timestamp_us"], len(example["frame_index"])
    labels = example["label_ids"].tolist()
    lengths = {len(left), len(right), len(lmask), len(rmask), len(stamps), T}
    if len(lengths) != 1:
        return [("sequence and mask lengths agree", False, f"array lengths found: {sorted(lengths)}")]
    known = np.ones(left.shape[1:], bool)
    if example["layout"] == LAYOUT_PARTIAL:
        known[7, 2] = False                    # z of landmark 7 is not stored

    def coordinates_agree_with_mask(points, mask):
        return (np.isfinite(points[mask][:, known]).all() and np.isnan(points[~mask]).all())

    used = np.concatenate([left[lmask][:, known], right[rmask][:, known]])
    return [
        ("sequence and mask lengths agree", True, f"{T} frames in every array"),
        ("frames are in order", bool(np.all(np.diff(example["frame_index"]) == 1)
                                     and np.all(np.diff(stamps) > 0)), "frame numbers and times increase"),
        ("mask matches the coordinates",
         bool(coordinates_agree_with_mask(left, lmask) and coordinates_agree_with_mask(right, rmask)),
         "numbers where detected, NaN elsewhere"),
        ("usable hand coordinates exist", bool(used.size and np.abs(used).max() < 5),
         f"{hand_frame_count(example)} frames with a hand"),
        ("enough hand frames for the letters", hand_frame_count(example) >= vocab.required_frames(labels),
         f"needs at least {vocab.required_frames(labels)}"),
        ("answer matches the phrase", bool(example["answer"] == training_answer(example["phrase"])
                                           and vocab.decode_ids(labels) == example["answer"]
                                           and len(labels) == len(example["phrase"])),
         f"{example['phrase']} -> {example['answer']}, repeated letters kept"),
        ("official split is train", example["split"] == "train", example["split"]),
    ]


def model_sentence(example):
    """The one-sentence meaning of an exported file for the next milestone."""
    return (f"For each of its {len(example['frame_index'])} frames the model reads the numbers in "
            "left_hand, right_hand, left_mask and right_mask (with timing from timestamp_us), and "
            f"from those it should predict the letters {'-'.join(example['answer'])} in that order.")


def plot_example(example, panels=6):
    """Hand movement from an exported file: poses over time, then wrist paths."""
    import matplotlib.pyplot as plt

    colors = {"left": "#d1495b", "right": "#00798c"}
    ids = example["landmark_ids"].tolist()
    bones = [(ids.index(a), ids.index(b)) for a, b in HAND_BONES if a in ids and b in ids]
    any_hand = np.flatnonzero(example["left_mask"] | example["right_mask"])
    chosen = any_hand[np.linspace(0, len(any_hand) - 1, min(panels, len(any_hand))).astype(int)]
    seconds = (example["timestamp_us"] - example["timestamp_us"][0]) / 1e6
    hands = [h for h in colors if example[f"{h}_mask"].any()]
    seen = np.concatenate([example[f"{h}_hand"][example[f"{h}_mask"]][:, :, :2].reshape(-1, 2) for h in hands])
    (x0, y0), (x1, y1) = np.nanmin(seen, axis=0) - 0.03, np.nanmax(seen, axis=0) + 0.03
    fig = plt.figure(figsize=(2.2 * max(len(chosen), 3), 6))
    grid = fig.add_gridspec(2, len(chosen))
    for col, t in enumerate(chosen):
        ax = fig.add_subplot(grid[0, col])
        for hand in hands:
            if example[f"{hand}_mask"][t]:
                p = example[f"{hand}_hand"][t]
                for a, b in bones:
                    ax.plot(p[[a, b], 0], p[[a, b], 1], color=colors[hand], lw=2)
                ax.scatter(p[:, 0], p[:, 1], color=colors[hand], s=12, label=hand)
        ax.set(title=f"frame {t}, {seconds[t]:.1f} s", xlim=(x0, x1), ylim=(y1, y0), xticks=[], yticks=[])
        ax.set_aspect(example["image_height"] / example["image_width"]
                      if example["image_width"] > 0 else 1)
        if col == 0:
            ax.legend(fontsize=8, loc="lower right")
    ax = fig.add_subplot(grid[1, :])
    for hand in hands:
        ax.plot(seconds, example[f"{hand}_hand"][:, 0, 0], ".-", color=colors[hand], ms=3, label=f"{hand} wrist x")
        ax.plot(seconds, example[f"{hand}_hand"][:, 0, 1], ".:", color=colors[hand], ms=3, label=f"{hand} wrist y")
    ax.set(xlabel="seconds", ylabel="position (fraction of image)",
           title="Wrist position over time (gaps = no hand detected)")
    ax.legend(fontsize=8, ncol=2)
    fig.suptitle(f"{example['answer']}  ({example['clip_id']})")
    fig.tight_layout()
    return fig


# ===================================================================== export
INDEX_COLUMNS = ("file", "clip_id", "video_name", "signer_id", "split", "phrase", "answer", "num_letters",
                 "num_frames", "frames_left_hand", "frames_right_hand", "frames_any_hand",
                 "duration_s", "frame_rate", "image_width", "image_height", "landmarks_per_hand",
                 "layout", "source_shard", "source_record")


def index_row(example, file_name):
    stamps = example["timestamp_us"]
    return {"file": file_name, "clip_id": example["clip_id"], "video_name": example["video_name"],
            "signer_id": example["signer_id"], "split": example["split"], "phrase": example["phrase"],
            "answer": example["answer"], "num_letters": len(example["answer"]),
            "num_frames": len(stamps), "frames_left_hand": int(example["left_mask"].sum()),
            "frames_right_hand": int(example["right_mask"].sum()),
            "frames_any_hand": hand_frame_count(example),
            "duration_s": round(float(stamps[-1] - stamps[0]) / 1e6, 3),
            "frame_rate": round(example["frame_rate"], 3), "image_width": example["image_width"],
            "image_height": example["image_height"],
            "landmarks_per_hand": int(len(example["landmark_ids"])), "layout": example["layout"],
            "source_shard": example["source_shard"], "source_record": example["source_record"]}


def read_index(out_dir):
    path = Path(out_dir) / "index.csv"
    if not path.exists():
        return []
    with open(path, newline="", encoding="utf-8") as f:
        return list(csv.DictReader(f))


def write_index(out_dir, rows):
    path = Path(out_dir) / "index.csv"
    tmp = path.with_name("index.csv.tmp")
    with open(tmp, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=INDEX_COLUMNS)
        writer.writeheader()
        writer.writerows(rows)
    tmp.replace(path)
    return path


def export_from_shard(shard_path, records, out_dir, target=20, shard_name=None,
                      min_letters=MIN_LETTERS, max_letters=MAX_LETTERS, skip_sensitive=True, log=print):
    """Read a downloaded TRAINING shard and export up to target usable recordings.

    Reading stops as soon as the target is reached. Running it again, or on
    another shard, keeps what was already exported. Returns a report dictionary.
    """
    shard_name = shard_name or str(shard_path)
    split = shard_split(shard_name)
    if split != "train":
        raise FsboardError(f"{shard_name} belongs to the official {split} split.\nWhat to check: this "
                           "milestone exports training data only. Validation and test stay untouched "
                           "for later evaluation.")
    out_dir = Path(out_dir)
    sequences = out_dir / "sequences"
    sequences.mkdir(parents=True, exist_ok=True)
    (out_dir / "README.txt").write_text(FORMAT_NOTES, encoding="utf-8")
    index = metadata_index(records)
    rows = read_index(out_dir)
    done = {r["clip_id"] for r in rows}
    reader = open_shard(shard_path)
    skipped, layouts = Counter(), Counter()
    exported = []
    read = paired = 0
    total = reader.num_record_batches
    for number, serialized in enumerate(iter_serialized(reader)):
        if len(rows) >= target:
            break
        read += 1
        if read % 100 == 0:
            log(f"   read {read} of {total} records, {len(rows)} sequences exported so far")
        example = parse_example(serialized)
        context = record_context(example)
        row, mismatch = match_metadata(context, index)
        if row is None:
            skipped[mismatch] += 1
            continue
        paired += 1
        reason = phrase_problem(row["phrase"], min_letters, max_letters)
        if not reason and row["signingType"] != "fs":
            reason = "not fingerspelling"
        if not reason and skip_sensitive and row["sensitiveContent"]:
            reason = "tagged sensitiveContent"
        if not reason and row["clipFilename"] in done:
            reason = "already exported earlier"
        if not reason:
            try:
                hands = decode_hands(example)
            except (ValueError, IndexError) as e:
                reason = f"unreadable landmarks ({e})"
        if not reason:
            built = build_example(context, row, hands, shard_name, number)
            needed = vocab.required_frames(built["label_ids"].tolist())
            layouts[built["layout"]] += 1
            if hand_frame_count(built) == 0:
                reason = "no hand detected in any frame"
            elif hand_frame_count(built) < needed:
                reason = "too few hand frames for its letters"
        if reason:
            skipped[reason] += 1
            continue
        path = save_example(built, sequences)
        rows.append(index_row(built, f"sequences/{path.name}"))
        done.add(built["clip_id"])
        exported.append(built["clip_id"])
        write_index(out_dir, rows)
    report = {"shard": Path(str(shard_name)).name, "records_in_shard": total, "records_read": read,
              "stopped_early": read < total, "paired_with_metadata": paired,
              "exported_now": exported, "exported_total": len(rows), "target": target,
              "skipped": dict(skipped.most_common()), "layouts": dict(layouts)}
    write_index(out_dir, rows)
    (out_dir / "export_report.json").write_text(json.dumps(report, indent=2), encoding="utf-8")
    return report


def report_text(report, eligible_total):
    """A plain-English summary. If short of the target, it says how many and why
    BEFORE suggesting another download; nothing more is downloaded by itself."""
    lines = [f"Shard {report['shard']}: read {report['records_read']} of {report['records_in_shard']} records"
             + (" (stopped once the target was reached)" if report["stopped_early"] else "") + ".",
             f"Pairing check: {report['paired_with_metadata']} of {report['records_read']} records matched "
             "the metadata by clip name, phrase and signer.",
             f"Exported from this shard: {len(report['exported_now'])}. "
             f"Total exported: {report['exported_total']} of a target of {report['target']}."]
    if report["skipped"]:
        lines.append("Why the other records were skipped:")
        lines += [f"   {count:5d}  {reason}" for reason, count in report["skipped"].items()]
    if report["exported_total"] < report["target"]:
        per_shard = eligible_total / NUM_TRAIN_SHARDS
        missing = report["target"] - report["exported_total"]
        extra = int(np.ceil(missing / max(per_shard, 1e-9)))
        lines += [f"\nFewer than {report['target']} sequences: {report['exported_total']} found. Reason: only "
                  f"{eligible_total:,} training clips in the whole dataset have a phrase of "
                  f"{MIN_LETTERS}-{MAX_LETTERS} letters and nothing else, about {per_shard:.1f} per shard. "
                  "Most phrases contain digits, symbols or spaces.",
                  f"Reaching {report['target']} would take about {extra} more shards "
                  f"(roughly {extra * 0.62:.1f} GB in total) if they are typical. Nothing further was "
                  "downloaded. To add one, change SHARD_NUMBER and run section 3 again; what you "
                  "already exported is kept."]
    return "\n".join(lines)


def zip_outputs(out_dir, zip_path):
    """One small zip of the exported files, replacing any earlier zip."""
    out_dir, zip_path = Path(out_dir), Path(zip_path)
    made = shutil.make_archive(str(zip_path.with_suffix("")), "zip", root_dir=out_dir.parent,
                               base_dir=out_dir.name)
    return Path(made)
''',
    "asl/seq/fsboard_video.py": r'''"""
fsboard_video.py - All 21 hand points from FSboard's VIDEOS, ready for training.

FSboard's own landmark files hold only 8 of the 21 hand points (see fsboard.py),
so here we run MediaPipe Hands on the videos, the same tracker family the webcam
app uses. Used by notebooks/fsboard_tcn_ctc_colab.ipynb.

    metadata -> choose clips -> for each clip:
        download video to temporary disk -> check its MD5 -> turn it upright
        -> mirror it (exactly like asl/engine.py does with the webcam)
        -> MediaPipe Hands -> save a small .npz -> delete the video

What each saved file keeps: left and right hand as (frames, 21, 3), NaN where that
hand was not found, a mask for each hand, frame times, image size, and the clip's
phrase, signer and official split. x and y are fractions of the upright, mirrored
image. Nothing is normalised at this stage, so the features can change later without
touching the videos again.

Why MediaPipe runs in its own small Python environment: mediapipe 0.10.14 needs older
package versions than Colab's TensorFlow. ensure_mediapipe_env() creates a separate
Python 3.11 environment (the same versions as the laptop), and extraction runs there:

    python -m asl.seq.fsboard_video --clips clips.json --out DIR --tmp DIR
"""
import argparse
import html
import json
import os
import random
import shutil
import struct
import subprocess
import sys
import time
from collections import Counter
from pathlib import Path

import numpy as np

from . import fsboard as fb
from . import vocab
from ..features import hand_size, normalise_points
from ..text_builder import BuilderConfig, TextBuilder
from .ctc import character_error_rate, collapse_doubles, has_double_letter
from .features import CausalResampler, SequenceFeaturizer, canonical_points

SPLITS = ("train", "validation", "test")
FORMAT_VERSION = "fsboard-video-npz-v1"
LETTER_LIMITS = (2, 100)          # letters-only phrases of any sensible length
MEDIAPIPE_VERSION = "0.10.14"     # same as requirements.txt
MEDIAPIPE_PYTHON = "3.11"         # same as the laptop
FEATURE_SETTINGS = {
    "aspect_correction": True,    # see isotropic_points()
    "trim_to_annotation": False,  # True = drop the idle moments before and after signing
}


def metadata_path(split):
    return f"daun_v3/metadata/daun_v3-{split}.json"


def clip_path(row):
    return f"daun_v3/video_clips/daun_v3-{row['split']}/{row['filename']}"


# ============================================================ choosing clips
def select_clips(records, split, max_clips=0, seed=42, min_letters=LETTER_LIMITS[0],
                 max_letters=LETTER_LIMITS[1], skip_sensitive=True, phrases=False):
    """Usable clips of one official split, plus a count of why others were skipped.

    phrases=False: one-word clips (letters only), as before Phase 2B.
    phrases=True:  ONLY clips whose phrase has words separated by spaces, such as
                   "new york"; one-word clips are left to the other call, so the two
                   groups never overlap.
    max_clips = 0 means all of them. The same seed always gives the same clips.
    """
    rows, skipped = fb.eligible_rows(records, min_letters, max_letters, skip_sensitive, allow_spaces=phrases)
    if phrases:
        one_word = [r for r in rows if " " not in r["phrase"]]
        rows = [r for r in rows if " " in r["phrase"]]
        if one_word:
            skipped["one word (chosen with the letters-only clips)"] += len(one_word)
    index = fb.metadata_index(records)
    for row in rows:
        r = index[row["filename"]]
        row["split"] = split
        row["clip_start_s"] = r.get("clipStartTimeS")
        row["annotation_start_s"] = r.get("annotationStartTimeS")
        row["annotation_end_s"] = r.get("annotationEndTimeS")
    return limit_clips(rows, max_clips, seed), skipped


def limit_clips(rows, max_clips=0, seed=42):
    """A repeatable random subset, in filename order. 0 keeps them all."""
    rows = sorted(rows, key=lambda r: r["filename"])
    if max_clips and len(rows) > max_clips:
        rows = sorted(random.Random(seed).sample(rows, max_clips), key=lambda r: r["filename"])
    return rows


def drop_seen_signers(rows, seen_signers):
    """Evaluation clips from people the model never trained on, and how many were removed."""
    kept = [r for r in rows if r["signer_id"] not in seen_signers]
    return kept, len(rows) - len(kept)


def html_table(rows, columns=None):
    """A small readable table for the notebook (list of dicts -> HTML)."""
    if not rows:
        return "<p><i>(nothing to show)</i></p>"
    columns = columns or list(rows[0])
    head = "".join(f"<th style='text-align:left;padding:2px 10px'>{html.escape(str(c))}</th>" for c in columns)
    body = "".join("<tr>" + "".join(f"<td style='padding:2px 10px'>{html.escape(str(r.get(c, '')))}</td>"
                                    for c in columns) + "</tr>" for r in rows)
    return f"<table><tr>{head}</tr>{body}</table>"


# ======================================================= reading video frames
def mp4_rotation(path):
    """(degrees clockwise to display upright, coded width, coded height) from an MP4.

    Many FSboard clips are stored sideways with a rotation flag, which browsers obey
    and OpenCV may not. Returns (0, 0, 0) if the file has no readable flag.
    """
    def boxes(buf, start, end):
        while start + 8 <= end:
            size, kind = struct.unpack(">I4s", buf[start:start + 8])
            head = 8
            if size == 1:
                size = struct.unpack(">Q", buf[start + 8:start + 16])[0]
                head = 16
            elif size == 0:
                size = end - start
            if size < head:
                return
            yield kind, start + head, start + size
            start += size

    with open(path, "rb") as f:
        total = os.path.getsize(path)
        pos = 0
        moov = None
        while pos + 8 <= total and moov is None:
            f.seek(pos)
            head = f.read(16)
            size, kind = struct.unpack(">I4s", head[:8])
            skip = 8
            if size == 1:
                size = struct.unpack(">Q", head[8:16])[0]
                skip = 16
            elif size == 0:
                size = total - pos
            if size < skip:
                break
            if kind == b"moov":
                f.seek(pos + skip)
                moov = f.read(size - skip)
            pos += size
    if moov is None:
        return 0, 0, 0
    for kind, a, b in boxes(moov, 0, len(moov)):
        if kind != b"trak":
            continue
        for sub, c, _ in boxes(moov, a, b):
            if sub != b"tkhd":
                continue
            matrix_at = c + (52 if moov[c] == 1 else 40)
            m = struct.unpack(">9i", moov[matrix_at:matrix_at + 36])
            width, height = struct.unpack(">II", moov[matrix_at + 36:matrix_at + 44])
            if not width or not height:
                continue                       # not the video track
            turn = (round(m[0] / 65536), round(m[1] / 65536), round(m[3] / 65536), round(m[4] / 65536))
            degrees = {(1, 0, 0, 1): 0, (0, 1, -1, 0): 90, (-1, 0, 0, -1): 180, (0, -1, 1, 0): 270}.get(turn, 0)
            return degrees, width >> 16, height >> 16
    return 0, 0, 0


def upright(frame, degrees, coded_w, coded_h):
    """Apply the stored rotation, unless the video reader already did."""
    import cv2

    h, w = frame.shape[:2]
    if degrees in (90, 270) and coded_w != coded_h and (w, h) == (coded_h, coded_w):
        return frame                           # already turned by the reader
    code = {90: cv2.ROTATE_90_CLOCKWISE, 180: cv2.ROTATE_180, 270: cv2.ROTATE_90_COUNTERCLOCKWISE}.get(degrees)
    return frame if code is None else cv2.rotate(frame, code)


def landmarks_from_video(path, detection=0.5, tracking=0.5, long_side=960):
    """Every frame of one video -> both hands as (T, 21, 3) with masks and times.

    Each frame is turned upright and mirrored before tracking, like the webcam in
    asl/engine.py, so "left" and "right" mean the same as in the live app. Frames are
    shrunk first only for speed: the results are fractions of the image.
    """
    import cv2
    import mediapipe as mp

    degrees, coded_w, coded_h = mp4_rotation(path)
    cap = cv2.VideoCapture(str(path))
    if not cap.isOpened():
        raise ValueError("OpenCV could not open this video")
    if hasattr(cv2, "CAP_PROP_ORIENTATION_AUTO"):
        cap.set(cv2.CAP_PROP_ORIENTATION_AUTO, 0)
    fps = float(cap.get(cv2.CAP_PROP_FPS) or 0)
    found = {"left": [], "right": []}
    stamps = []
    size = None
    with mp.solutions.hands.Hands(static_image_mode=False, max_num_hands=2,
                                  min_detection_confidence=detection,
                                  min_tracking_confidence=tracking) as tracker:
        while True:
            ok, frame = cap.read()
            if not ok:
                break
            stamp = int(round(cap.get(cv2.CAP_PROP_POS_MSEC) * 1000))
            stamps.append(max(stamp, stamps[-1] + 1) if stamps else stamp)
            frame = cv2.flip(upright(frame, degrees, coded_w, coded_h), 1)
            h, w = frame.shape[:2]
            size = (w, h)
            scale = long_side / max(h, w)
            if scale < 1:
                frame = cv2.resize(frame, (round(w * scale), round(h * scale)), interpolation=cv2.INTER_AREA)
            result = tracker.process(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
            best = {}
            for landmarks, handed in zip(result.multi_hand_landmarks or [], result.multi_handedness or []):
                label, score = handed.classification[0].label.lower(), handed.classification[0].score
                if label not in best or score > best[label][0]:
                    best[label] = (score, np.array([[p.x, p.y, p.z] for p in landmarks.landmark], np.float32))
            for hand in found:
                found[hand].append(best[hand][1] if hand in best else None)
    cap.release()
    if not stamps:
        raise ValueError("no frames could be read from this video")
    T = len(stamps)
    out = {"timestamp_us": np.array(stamps, np.int64), "frame_index": np.arange(T, dtype=np.int32),
           "frame_rate": fps, "image_width": size[0], "image_height": size[1]}
    for hand, frames in found.items():
        arr = np.full((T, 21, 3), np.nan, np.float32)
        for t, points in enumerate(frames):
            if points is not None:
                arr[t] = points
        out[f"{hand}_hand"] = arr
        out[f"{hand}_mask"] = np.array([p is not None for p in frames], bool)
    return out


def build_recording(row, landmarks):
    """Everything saved for one clip. Its answer comes from its own metadata row."""
    answer = fb.training_answer(row["phrase"])
    start = row.get("clip_start_s") or 0.0
    return dict(landmarks,
                clip_id=row["filename"], video_name=row["filename"], signer_id=row["signer_id"],
                split=row["split"], phrase=row["phrase"], answer=answer,
                label_ids=np.array(vocab.encode(answer), np.int32),
                landmark_ids=np.arange(21, dtype=np.int32), layout=fb.LAYOUT_COMPLETE,
                mirrored_input=True,
                annotation_start_s=float((row.get("annotation_start_s") or start) - start),
                annotation_end_s=float((row.get("annotation_end_s") or start) - start),
                source_shard="video", source_record=-1, format_version=FORMAT_VERSION)


def recording_path(row, out_dir):
    return Path(out_dir) / row["split"] / (Path(row["filename"]).stem + ".npz")


def save_recording(recording, path):
    """Appears under its final name only when complete, so a rerun can trust it."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.stem + ".tmp.npz")
    np.savez_compressed(tmp, **recording)
    tmp.replace(path)


def extract_clips(clips, out_dir, tmp_dir, detection=0.5, tracking=0.5, log=print, stop_after_failures=5):
    """Download, track, save and delete, one clip at a time. Finished clips are skipped."""
    out_dir, tmp_dir = Path(out_dir), Path(tmp_dir)
    tmp_dir.mkdir(parents=True, exist_ok=True)
    report = {"total": len(clips), "already_done": 0, "done": 0, "failed": [], "seconds": 0.0, "stopped": ""}
    started = time.time()
    in_a_row = 0
    for i, row in enumerate(clips, 1):
        out = recording_path(row, out_dir)
        if out.exists():
            report["already_done"] += 1
            continue
        video = tmp_dir / row["filename"]
        try:
            fb.download(clip_path(row), video, row["size_bytes"], row["md5"], log=lambda *_: None)
            recording = build_recording(row, landmarks_from_video(video, detection, tracking))
            save_recording(recording, out)
            in_a_row = 0
            report["done"] += 1
            if report["done"] <= 5 or report["done"] % 25 == 0:
                done = report["done"]
                left = (time.time() - started) / done * (len(clips) - i)
                hands = int(np.sum(recording["left_mask"] | recording["right_mask"]))
                log(f"[{i}/{len(clips)}] {row['answer']}: {len(recording['frame_index'])} frames, "
                    f"hand in {hands}; about {left / 60:.0f} min left")
        except Exception as problem:                       # one bad clip must not end the run
            in_a_row += 1
            report["failed"].append({"clip": row["filename"], "problem": str(problem)[:300]})
            log(f"[{i}/{len(clips)}] {row['filename']} FAILED: {str(problem).splitlines()[0][:150]}")
            if in_a_row >= stop_after_failures:
                report["stopped"] = str(problem)
                log(f"Stopping: {in_a_row} clips in a row failed. The last problem was:\n{problem}")
                break
        finally:
            video.unlink(missing_ok=True)
            video.with_name(video.name + ".part").unlink(missing_ok=True)
    report["seconds"] = round(time.time() - started, 1)
    return report


# =================================== running MediaPipe in its own environment
def check_mediapipe(python_exe):
    """Versions from that Python, or a plain-English error."""
    code = ("import json, cv2, numpy, mediapipe as mp; mp.solutions.hands; "
            "print(json.dumps({'mediapipe': mp.__version__, 'opencv': cv2.__version__, 'numpy': numpy.__version__}))")
    r = subprocess.run([str(python_exe), "-c", code], capture_output=True, text=True,
                       env=dict(os.environ, GLOG_minloglevel="3"))
    if r.returncode:
        raise fb.FsboardError(f"MediaPipe could not start in {python_exe}.\nWhat to check: delete that folder in the "
                              f"Files panel and run this cell again.\nDetails: {r.stderr.strip()[-500:]}")
    return json.loads(r.stdout.strip().splitlines()[-1])


def ensure_mediapipe_env(env_dir, log=print):
    """A separate Python 3.11 + mediapipe 0.10.14 environment. Returns its python."""
    env_dir = Path(env_dir)
    python_exe = env_dir / ("Scripts/python.exe" if os.name == "nt" else "bin/python")
    if python_exe.exists():
        try:
            log(f"MediaPipe environment already here: {check_mediapipe(python_exe)}")
            return python_exe
        except fb.FsboardError:
            shutil.rmtree(env_dir, ignore_errors=True)
    log(f"Installing Python {MEDIAPIPE_PYTHON} and mediapipe {MEDIAPIPE_VERSION} in their own folder ...")
    uv = shutil.which("uv")
    commands = [] if uv else [[sys.executable, "-m", "pip", "install", "-q", "uv"]]
    uv = uv or "uv"
    commands += [[uv, "venv", "--python", MEDIAPIPE_PYTHON, str(env_dir)],
                 [uv, "pip", "install", "-q", "--python", str(python_exe), f"mediapipe=={MEDIAPIPE_VERSION}", "numpy<2"]]
    for command in commands:
        r = subprocess.run(command, capture_output=True, text=True)
        if r.returncode:
            raise fb.FsboardError(f"Installing MediaPipe failed at: {' '.join(command[:3])}\nWhat to check: is this Colab "
                                  f"session still connected? Run the cell again.\nDetails: {r.stderr.strip()[-500:]}")
    log(f"Installed: {check_mediapipe(python_exe)}")
    return python_exe


def run_extraction(clips, out_dir, tmp_dir, python_exe, code_dir, log=print, detection=0.5, tracking=0.5):
    """Run extract_clips() in the MediaPipe environment and pass its progress along."""
    tmp_dir = Path(tmp_dir)
    tmp_dir.mkdir(parents=True, exist_ok=True)
    clips_file, report_file, errors_file = (tmp_dir / n for n in ("clips.json", "report.json", "errors.txt"))
    clips_file.write_text(json.dumps(clips), encoding="utf-8")
    report_file.unlink(missing_ok=True)
    env = dict(os.environ, PYTHONPATH=str(code_dir), PYTHONUNBUFFERED="1", GLOG_minloglevel="3",
               TF_CPP_MIN_LOG_LEVEL="3")
    command = [str(python_exe), "-m", "asl.seq.fsboard_video", "--clips", str(clips_file), "--out", str(out_dir),
               "--tmp", str(tmp_dir), "--report", str(report_file), "--base-url", fb.BASE_URL,
               "--detection", str(detection), "--tracking", str(tracking)]
    with open(errors_file, "w", encoding="utf-8") as errors:
        process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=errors, text=True, env=env)
        for line in process.stdout:
            log(line.rstrip())
        code = process.wait()
    if code or not report_file.exists():
        tail = errors_file.read_text(encoding="utf-8", errors="replace")[-800:]
        raise fb.FsboardError("Hand extraction stopped unexpectedly.\nWhat to check: run this cell again; finished "
                              f"clips are kept and skipped.\nDetails: {tail}")
    return json.loads(report_file.read_text(encoding="utf-8"))


# ======================================================== recordings -> features
def isotropic_points(points, width, height):
    """Undo stretching from the image's shape, so a hand has the same proportions in a
    portrait phone video and a landscape webcam.

    x and y are fractions of the image's width and height, which are different lengths.
    Here both become fractions of the LONGER side, measured about the image's centre.
    """
    p = np.array(points, np.float32, copy=True)
    longest = float(max(width, height))
    p[..., 0] = (p[..., 0] - 0.5) * width / longest + 0.5
    p[..., 1] = (p[..., 1] - 0.5) * height / longest + 0.5
    p[..., 2] = p[..., 2] * width / longest    # MediaPipe's z is on the same scale as x
    return p


def signing_hand(recording):
    """One hand per recording: the one found in more frames (ties: right)."""
    return "right" if recording["right_mask"].sum() >= recording["left_mask"].sum() else "left"


def recording_features(recording, feature_cfg):
    """A saved recording -> UNnormalised (T, 68) features at target_fps."""
    hand = signing_hand(recording)
    points = recording[f"{hand}_hand"]
    if feature_cfg.get("aspect_correction"):
        points = isotropic_points(points, recording["image_width"], recording["image_height"])
    points = canonical_points(points, hand)
    mask = recording[f"{hand}_mask"]
    seconds = (recording["timestamp_us"] - recording["timestamp_us"][0]) / 1e6
    keep = np.ones(len(seconds), bool)
    if feature_cfg.get("trim_to_annotation"):
        keep = (seconds >= recording["annotation_start_s"]) & (seconds <= recording["annotation_end_s"])
    resampler = CausalResampler(feature_cfg["target_fps"])
    featurizer = SequenceFeaturizer(feature_cfg)
    rows = []
    for t in np.flatnonzero(keep):
        for q in resampler.push(float(seconds[t]), points[t] if mask[t] else None):
            rows.append(featurizer.step(q))
    return np.stack(rows) if rows else np.zeros((0, 68), np.float32)


def build_examples(rows, out_dir, feature_cfg, max_frames=1500):
    """Saved recordings -> training examples. Unusable ones are excluded whole, with a reason."""
    examples, excluded = [], Counter()
    for row in rows:
        path = recording_path(row, out_dir)
        if not path.exists():
            excluded["no saved file (extraction failed)"] += 1
            continue
        recording = fb.load_example(path)
        labels = recording["label_ids"].tolist()
        needed = vocab.required_frames(labels)
        hands = int(recording[f"{signing_hand(recording)}_mask"].sum())
        if hands == 0:
            excluded["no hand found in any frame"] += 1
            continue
        if hands < needed:
            excluded["too few hand frames for its letters"] += 1
            continue
        x = recording_features(recording, feature_cfg)
        if len(x) < needed:
            excluded["too short for its letters"] += 1
        elif len(x) > max_frames:
            excluded["too long"] += 1
        else:
            examples.append({"x": x, "y": recording["label_ids"], "text": recording["answer"],
                             "sequence_id": recording["clip_id"], "signer": recording["signer_id"],
                             "split": recording["split"], "path": str(path),
                             "hand_share": hands / len(recording["frame_index"])})
    return examples, excluded


def check_recording(recording):
    """fsboard.check_example, for any official split, plus a check for all 21 points."""
    results = [r for r in fb.check_example(dict(recording, split="train")) if r[0] != "official split is train"]
    results.append(("all 21 hand points", recording["left_hand"].shape[1:] == (21, 3),
                    f"shape {recording['left_hand'].shape}"))
    return results


# ================================================================== evaluation
def letter_report(refs, hyps):
    """For each true letter: how often it was right, replaced by another, or missed.
    Also the most common confusions (true letter -> predicted letter)."""
    stats = {}
    pairs = Counter()
    for ref, hyp in zip(refs, hyps):
        n, m = len(ref), len(hyp)
        d = np.zeros((n + 1, m + 1), np.int32)
        d[:, 0], d[0, :] = np.arange(n + 1), np.arange(m + 1)
        for i in range(1, n + 1):
            for j in range(1, m + 1):
                d[i, j] = min(d[i - 1, j] + 1, d[i, j - 1] + 1, d[i - 1, j - 1] + (ref[i - 1] != hyp[j - 1]))
        i, j = n, m
        while i > 0 or j > 0:
            if i > 0 and j > 0 and d[i, j] == d[i - 1, j - 1] + (ref[i - 1] != hyp[j - 1]):
                s = stats.setdefault(ref[i - 1], Counter())
                s["seen"] += 1
                if ref[i - 1] == hyp[j - 1]:
                    s["right"] += 1
                else:
                    s["replaced"] += 1
                    pairs[f"{_shown(ref[i - 1])} -> {_shown(hyp[j - 1])}"] += 1
                i, j = i - 1, j - 1
            elif j > 0 and d[i, j] == d[i, j - 1] + 1:
                j -= 1
            else:
                s = stats.setdefault(ref[i - 1], Counter())
                s["seen"] += 1
                s["missed"] += 1
                i -= 1
    rows = [{"letter": "space" if c == " " else c, "seen": s["seen"], "right": s["right"], "replaced": s["replaced"], "missed": s["missed"],
             "accuracy": round(s["right"] / s["seen"], 3)} for c, s in stats.items()]
    rows.sort(key=lambda r: (r["accuracy"], r["letter"]))
    return {"letters": rows, "confusions": [{"pair": p, "count": c} for p, c in pairs.most_common(15)]}


def _shown(c):
    return "space" if c == " " else c


def _edit_distance(a, b):
    """Edits (insert, delete, replace) between two sequences of letters or words."""
    row = list(range(len(b) + 1))
    for i, x in enumerate(a, 1):
        prev, row[0] = row[0], i
        for j, y in enumerate(b, 1):
            prev, row[j] = row[j], min(row[j] + 1, row[j - 1] + 1, prev + (x != y))
    return row[-1]


def _matched_spaces(ref, hyp):
    """Spaces of ref that the best alignment pairs with a space in hyp."""
    n, m = len(ref), len(hyp)
    d = np.zeros((n + 1, m + 1), np.int32)
    d[:, 0], d[0, :] = np.arange(n + 1), np.arange(m + 1)
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            d[i, j] = min(d[i - 1, j] + 1, d[i, j - 1] + 1, d[i - 1, j - 1] + (ref[i - 1] != hyp[j - 1]))
    i, j, found = n, m, 0
    while i > 0 and j > 0:
        if d[i, j] == d[i - 1, j - 1] + (ref[i - 1] != hyp[j - 1]):
            found += ref[i - 1] == " " == hyp[j - 1]
            i, j = i - 1, j - 1
        elif d[i, j] == d[i, j - 1] + 1:
            j -= 1
        else:
            i -= 1
    return found


def phrase_scores(refs, hyps):
    """Scores for phrases with spaces (Phase 2B), next to the usual letter scores.

    letters_only_error_rate  wrong letters when spaces are ignored on both sides,
                             comparable with the one-word models
    word_error_rate          wrong, missing or extra WORDS per true word
    spaces_found             share of true spaces the model also wrote, in the right place
    extra_spaces             spaces the model wrote that are not in the truth
    """
    if not refs:
        return {"recordings": 0}
    plain = [r.replace(" ", "") for r in refs], [h.replace(" ", "") for h in hyps]
    true_spaces = sum(r.count(" ") for r in refs)
    matched = sum(_matched_spaces(r, h) for r, h in zip(refs, hyps))
    words = sum(len(r.split()) for r in refs)
    return {"recordings": len(refs),
            "character_error_rate": round(character_error_rate(refs, hyps), 4),
            "letters_only_error_rate": round(character_error_rate(*plain), 4),
            "word_error_rate": round(sum(_edit_distance(r.split(), h.split()) for r, h in zip(refs, hyps))
                                     / max(1, words), 4),
            "exact_phrase_accuracy": round(float(np.mean([r == h for r, h in zip(refs, hyps)])), 4),
            "true_spaces": true_spaces,
            "spaces_found": round(matched / true_spaces, 4) if true_spaces else None,
            "extra_spaces": int(sum(h.count(" ") for h in hyps) - matched)}


def score_texts(refs, hyps):
    """The same scores for any model, so two models can be compared fairly."""
    doubled = [k for k, r in enumerate(refs) if has_double_letter(r)]
    return {"recordings": len(refs),
            "character_error_rate": round(character_error_rate(refs, hyps), 4) if refs else None,
            "exact_word_accuracy": round(float(np.mean([r == h for r, h in zip(refs, hyps)])), 4) if refs else None,
            "double_letter_words": len(doubled),
            "double_letter_word_accuracy": round(float(np.mean([refs[k] == hyps[k] for k in doubled])), 4) if doubled else None,
            "doubles_collapsed": int(sum(hyps[k] == collapse_doubles(refs[k]) for k in doubled))}


def baseline_text(recording, predict, labels, aspect_correction=True):
    """What the baseline app would write for this recording: letter per frame, then
    the app's own hold-still rules, at its default settings. Spaces are not scored.

    predict: function from (N, 63) features to (N, classes) probabilities.
    """
    hand = signing_hand(recording)
    points, mask = recording[f"{hand}_hand"], recording[f"{hand}_mask"]
    if aspect_correction:
        points = isotropic_points(points, recording["image_width"], recording["image_height"])
    frames = np.flatnonzero(mask)
    builder = TextBuilder(BuilderConfig(auto_space_seconds=0, control_gestures=False))
    probabilities = {}
    if len(frames):
        scores = np.asarray(predict(np.stack([normalise_points(points[t]) for t in frames])))
        probabilities = dict(zip(frames.tolist(), scores))
    for t, stamp in enumerate(recording["timestamp_us"].tolist()):
        if t in probabilities:
            p = probabilities[t]
            best = int(np.argmax(p))
            builder.update(stamp / 1e6, labels[best], float(p[best]),
                           wrist=(float(points[t, 0, 0]), float(points[t, 0, 1])), size=hand_size(points[t]))
        else:
            builder.update(stamp / 1e6, None)
    return "".join(c for c in builder.snapshot()["text"] if c.isalpha())


def main():
    ap = argparse.ArgumentParser(description="Extract 21 hand points from FSboard videos")
    ap.add_argument("--clips", required=True)
    ap.add_argument("--out", required=True)
    ap.add_argument("--tmp", required=True)
    ap.add_argument("--report", required=True)
    ap.add_argument("--base-url", default=fb.BASE_URL)
    ap.add_argument("--detection", type=float, default=0.5)
    ap.add_argument("--tracking", type=float, default=0.5)
    args = ap.parse_args()
    fb.BASE_URL = args.base_url
    clips = json.loads(Path(args.clips).read_text(encoding="utf-8"))
    report = extract_clips(clips, args.out, args.tmp, args.detection, args.tracking)
    Path(args.report).write_text(json.dumps(report, indent=2), encoding="utf-8")


if __name__ == "__main__":
    main()
''',
    "asl/seq/data.py": r'''"""
data.py - Google "ASL Fingerspelling" (Kaggle competition `asl-fingerspelling`)
metadata, pilot-subset selection, remote extraction, signer split and batching.

Not to be confused with Google's ISOLATED SIGNS dataset (Kaggle
`asl-signs`, 250 single word signs). That one has no fingerspelled text.

What the dataset gives us (verified in the notebook, not assumed):
    train.csv / supplemental_metadata.csv
        path, file_id, sequence_id, participant_id, phrase
    <path>.parquet   one row per (sequence_id, frame), MediaPipe Holistic
        landmarks as columns such as x_right_hand_0 ... z_left_hand_20
        (NaN when that hand was not detected). One parquet file holds MANY
        sequences, so we download one file, keep only the selected
        sequences' hand columns, save a compact shard and delete the file.
What it does NOT give: timestamps / frame rate, session or recording-device
IDs, video. These gaps are written into the manifest.

Heavy imports (pandas, pyarrow, kaggle) are done inside functions so the
rest of asl.seq works on the laptop without them.
"""
import json
import shutil
import zipfile
from pathlib import Path

import numpy as np

from . import vocab
from .fsboard import kaggle_auth_from_colab_secrets  # noqa: F401  (used by the notebook)
from .features import (SequenceFeaturizer, canonical_points, VALID, WRIST_VEL,
                       FINGERS, WRIST_POS, NUM_FEATURES)

COMPETITION = "asl-fingerspelling"
METADATA_FILES = ["train.csv", "supplemental_metadata.csv"]
HANDS = ("right", "left")
PROVENANCE_GAPS = [
    "No per-frame timestamps or frame rate in the dataset; target_fps is an assumption.",
    "No session or recording-device IDs; each sequence_id is treated as one recording.",
    "Camera image size/aspect ratio not given; landmark x and y are in normalised image units.",
    "Signing hand is chosen per sequence as the hand tracked in more frames.",
]


# ===================================================================== Kaggle
def kaggle_api():
    from kaggle.api.kaggle_api_extended import KaggleApi

    api = KaggleApi()
    api.authenticate()
    return api


def list_competition_files(api, competition=COMPETITION, page_size=200):
    """All files with their sizes in bytes (follows pagination)."""
    import pandas as pd

    rows, token = [], None
    while True:
        resp = api.competition_list_files(competition, page_token=token, page_size=page_size)
        for f in resp.files:
            size = getattr(f, "total_bytes", None)
            if size is None:
                size = getattr(f, "totalBytes", None)
            rows.append({"name": f.name, "bytes": int(size or 0)})
        token = getattr(resp, "next_page_token", None)
        if not token:
            break
    return pd.DataFrame(rows)


def download_competition_file(api, name, dest_dir, competition=COMPETITION):
    """Download ONE file into dest_dir (unzipping if Kaggle zipped it).
    Returns the local path of the file."""
    dest_dir = Path(dest_dir)
    dest_dir.mkdir(parents=True, exist_ok=True)
    target = dest_dir / name
    if target.exists():
        return target
    api.competition_download_file(competition, name, path=str(dest_dir), quiet=True)
    zipped = target.with_name(target.name + ".zip")
    if zipped.exists():
        with zipfile.ZipFile(zipped) as z:
            member = [m for m in z.namelist() if m.endswith(Path(name).name)][0]
            with z.open(member) as src, open(target, "wb") as dst:
                shutil.copyfileobj(src, dst)
        zipped.unlink()
    if not target.exists():
        raise FileNotFoundError(f"Download of {name} did not produce {target}")
    return target


def free_gb(path):
    return shutil.disk_usage(str(path)).free / 1e9


# ================================================================== metadata
def load_metadata(csv_paths):
    import pandas as pd

    frames = []
    for p in csv_paths:
        df = read_csv_text(p)
        df["source"] = Path(p).stem
        frames.append(df)
    meta = pd.concat(frames, ignore_index=True)
    return annotate_compatibility(meta)


def read_csv_text(path):
    """read_csv that keeps words such as "NA" or "NAN" as text (pandas would
    otherwise turn them into missing values)."""
    import pandas as pd

    return pd.read_csv(path, keep_default_na=False)


def read_manifest(path):
    return read_csv_text(path)


def _incompatibility(text):
    t = vocab.normalise_transcript(text)
    if not t:
        return "empty"
    if vocab.is_compatible(t):
        return ""
    if " " in t:
        return "space"
    if any(c.isdigit() for c in t):
        return "digit"
    return "other_symbol"


def annotate_compatibility(meta):
    meta = meta.copy()
    meta["transcript"] = meta["phrase"].astype(str).map(vocab.normalise_transcript)
    meta["exclude_reason"] = meta["phrase"].astype(str).map(_incompatibility)
    meta["compatible"] = meta["exclude_reason"] == ""
    meta["label_len"] = meta["transcript"].str.len()
    return meta


def select_pilot_subset(meta, file_sizes, budget_gb=5.0, max_sequences=3000,
                        min_signers=6, seed=0):
    """Choose which parquet files to download and which sequences to keep.

    Files are ranked by "compatible sequences per GB", then taken while the
    total download stays under budget_gb. Sequences are then sampled (with
    a fixed seed) up to max_sequences. Returns (selection DataFrame, summary).
    """
    ok = meta[meta["compatible"]]
    sizes = dict(zip(file_sizes["name"], file_sizes["bytes"]))
    per_file = ok.groupby("path").agg(n=("sequence_id", "size"),
                                      signers=("participant_id", "nunique")).reset_index()
    per_file["bytes"] = per_file["path"].map(sizes).fillna(0).astype(np.int64)
    per_file = per_file[per_file["bytes"] > 0]
    if per_file.empty:
        raise ValueError("No metadata 'path' matched a listed file name; compare "
                         "meta['path'].head() with file_sizes['name'].head()")
    per_file["per_gb"] = per_file["n"] / (per_file["bytes"] / 1e9)
    per_file = per_file.sort_values("per_gb", ascending=False)

    chosen, total = [], 0
    for r in per_file.itertuples():
        if total + r.bytes > budget_gb * 1e9:
            continue
        chosen.append(r.path)
        total += r.bytes
        if ok[ok["path"].isin(chosen)].shape[0] >= max_sequences * 1.2:
            break
    sel = ok[ok["path"].isin(chosen)]
    if len(sel) > max_sequences:
        sel = sel.sample(n=max_sequences, random_state=seed)
    sel = sel.sort_values(["path", "sequence_id"]).reset_index(drop=True)
    summary = {
        "budget_gb": budget_gb, "download_gb": round(total / 1e9, 3),
        "files": len(chosen), "sequences": int(len(sel)),
        "signers": int(sel["participant_id"].nunique()), "seed": seed,
        "max_sequences": max_sequences,
        "warning": (f"only {sel['participant_id'].nunique()} signers (< {min_signers}); "
                    "increase the budget for a meaningful signer split")
        if sel["participant_id"].nunique() < min_signers else "",
    }
    return sel, summary


# ================================================================ extraction
def hand_columns(hand):
    return [f"{a}_{hand}_hand_{i}" for i in range(21) for a in "xyz"]


def parquet_schema(path):
    import pyarrow.parquet as pq

    return pq.read_schema(str(path)).names


def read_hands(path, sequence_ids):
    """Read only frame + both hands' columns for the given sequences."""
    import pyarrow.parquet as pq

    cols = ["frame"] + hand_columns("right") + hand_columns("left")
    names = parquet_schema(path)
    missing = [c for c in cols if c not in names]
    if missing:
        raise KeyError(f"{path}: expected columns not found, e.g. {missing[:3]}")
    if "sequence_id" in names:
        table = pq.read_table(str(path), columns=["sequence_id"] + cols,
                              filters=[("sequence_id", "in", list(map(int, sequence_ids)))])
        df = table.to_pandas()
        if "sequence_id" not in df.columns:      # pandas metadata turned it back into the index
            df = df.reset_index()
    else:  # stored only as pandas index
        import pandas as pd
        df = pd.read_parquet(path, columns=cols)
        df = df[df.index.isin(sequence_ids)].reset_index()
    return df


def sequence_record(sequence_id, df_seq):
    """One sequence -> compact record with the signing hand only.

    Frames are sorted; missing frame numbers inside the range are inserted
    as NaN (invalid) rows so time stays evenly spaced.
    """
    df_seq = df_seq.sort_values("frame")
    frames = df_seq["frame"].to_numpy(np.int64)
    full = np.arange(frames.min(), frames.max() + 1)
    pos = np.searchsorted(full, frames)
    hands = {}
    counts = {}
    for h in HANDS:
        arr = df_seq[hand_columns(h)].to_numpy(np.float32).reshape(-1, 21, 3)
        pts = np.full((len(full), 21, 3), np.nan, np.float32)
        pts[pos] = arr
        hands[h] = pts
        counts[h] = int(np.all(np.isfinite(arr.reshape(len(arr), -1)), axis=1).sum())
    hand = "right" if counts["right"] >= counts["left"] else "left"
    return {"sequence_id": int(sequence_id), "hand": hand, "points": hands[hand],
            "frames": full.astype(np.int32), "valid_right": counts["right"],
            "valid_left": counts["left"], "inserted_frames": int(len(full) - len(frames)),
            "duplicate_frames": int(len(frames) - len(np.unique(frames)))}


def save_shard(path, records):
    """Compact .npz without pickled objects (safe to load anywhere)."""
    lengths = np.array([len(r["points"]) for r in records], np.int64)
    np.savez_compressed(
        path,
        sequence_id=np.array([r["sequence_id"] for r in records], np.int64),
        hand=np.array([r["hand"] for r in records]),
        lengths=lengths,
        points=np.concatenate([r["points"] for r in records]) if records else np.zeros((0, 21, 3), np.float32),
        frames=np.concatenate([r["frames"] for r in records]) if records else np.zeros(0, np.int32),
        valid_right=np.array([r["valid_right"] for r in records], np.int64),
        valid_left=np.array([r["valid_left"] for r in records], np.int64),
        inserted_frames=np.array([r["inserted_frames"] for r in records], np.int64),
        duplicate_frames=np.array([r["duplicate_frames"] for r in records], np.int64),
    )


def load_shard(path):
    z = np.load(path, allow_pickle=False)
    ends = np.cumsum(z["lengths"])
    starts = ends - z["lengths"]
    out = []
    for i, (a, b) in enumerate(zip(starts, ends)):
        out.append({"sequence_id": int(z["sequence_id"][i]), "hand": str(z["hand"][i]),
                    "points": z["points"][a:b], "frames": z["frames"][a:b],
                    "valid_right": int(z["valid_right"][i]), "valid_left": int(z["valid_left"][i]),
                    "inserted_frames": int(z["inserted_frames"][i]),
                    "duplicate_frames": int(z["duplicate_frames"][i])})
    return out


def shard_name(parquet_path):
    return str(parquet_path).replace("/", "__").replace(".parquet", ".npz")


def prepare_shards(api, selection, tmp_dir, shard_dir, keep_raw=False, log=print):
    """For each selected parquet file: download to TEMPORARY disk, extract the
    selected sequences, save a compact shard to PERSISTENT storage, delete
    the raw file. Already finished shards are skipped (safe to rerun)."""
    tmp_dir, shard_dir = Path(tmp_dir), Path(shard_dir)
    shard_dir.mkdir(parents=True, exist_ok=True)
    groups = selection.groupby("path")["sequence_id"].apply(list)
    for i, (path, ids) in enumerate(groups.items(), 1):
        out = shard_dir / shard_name(path)
        if out.exists():
            log(f"[{i}/{len(groups)}] {path}: shard exists, skipped")
            continue
        local = download_competition_file(api, path, tmp_dir)
        df = read_hands(local, ids)
        records = [sequence_record(sid, g) for sid, g in df.groupby("sequence_id")]
        found = {r["sequence_id"] for r in records}
        lost = sorted(set(map(int, ids)) - found)
        tmp_out = out.with_name(out.stem + ".tmp.npz")
        save_shard(tmp_out, records)
        tmp_out.replace(out)                      # appears only when complete
        if not keep_raw:
            local.unlink()
        log(f"[{i}/{len(groups)}] {path}: kept {len(records)} sequences "
            f"({out.stat().st_size / 1e6:.1f} MB)" + (f", {len(lost)} not found" if lost else "")
            + f"; temp disk free {free_gb(tmp_dir):.1f} GB")


def load_all_shards(shard_dir):
    recs = {}
    for p in sorted(Path(shard_dir).glob("*.npz")):
        if p.name.endswith(".tmp.npz"):
            continue
        for r in load_shard(p):
            recs[r["sequence_id"]] = r
    return recs


# ===================================================================== split
def split_by_signer(meta, val_frac=0.15, test_frac=0.15, seed=0):
    """Assign whole SIGNERS (participant_id) to train / val / test.

    A signer's recordings (and any augmented copies, which are only made on
    the fly for the training split) never appear in two splits.
    Returns {participant_id: split}.
    """
    signers = np.array(sorted(meta["participant_id"].unique()))
    rng = np.random.default_rng(seed)
    rng.shuffle(signers)
    n = len(signers)
    if n < 3:
        raise ValueError(f"Need at least 3 signers for a signer split, got {n}")
    n_test = max(1, int(round(n * test_frac)))
    n_val = max(1, int(round(n * val_frac)))
    out = {}
    for i, s in enumerate(signers):
        out[int(s)] = "test" if i < n_test else "val" if i < n_test + n_val else "train"
    return out


def write_manifests(folder, selection, signer_split, summary, examples_info=None):
    """Save what was selected and how it was split, so runs can be repeated."""
    folder = Path(folder)
    folder.mkdir(parents=True, exist_ok=True)
    sel = selection.copy()
    sel["split"] = sel["participant_id"].map(signer_split)
    sel.to_csv(folder / "selection_manifest.csv", index=False)
    info = dict(summary, provenance_gaps=PROVENANCE_GAPS,
                split_counts=sel["split"].value_counts().to_dict(),
                split_signers=sel.groupby("split")["participant_id"].nunique().to_dict())
    if examples_info:
        info["examples"] = examples_info
    (folder / "dataset_summary.json").write_text(json.dumps(info, indent=2, default=int))
    return sel


# ================================================================== examples
def record_features(record, feature_cfg=None):
    """Raw record -> UNnormalised (T, 68) features, using the shared featurizer."""
    pts = canonical_points(record["points"], record["hand"])
    return SequenceFeaturizer(feature_cfg).featurize(pts)


def build_examples(selection, records, feature_cfg=None, max_frames=1000):
    """Join manifest rows with extracted landmarks. Recordings that cannot be
    used are EXCLUDED (never truncated): too short for their transcript
    under CTC, too long, missing, or with no valid hand frame."""
    examples, excluded = [], {"missing": 0, "too_short_for_ctc": 0, "too_long": 0, "no_hand": 0}
    for row in selection.itertuples():
        rec = records.get(int(row.sequence_id))
        if rec is None:
            excluded["missing"] += 1
            continue
        y = vocab.encode(row.transcript)
        x = record_features(rec, feature_cfg)
        if x[:, VALID].sum() == 0:
            excluded["no_hand"] += 1
        elif len(x) < vocab.required_frames(y):
            excluded["too_short_for_ctc"] += 1
        elif len(x) > max_frames:
            excluded["too_long"] += 1
        else:
            examples.append({"x": x, "y": np.array(y, np.int32), "text": row.transcript,
                             "sequence_id": int(row.sequence_id),
                             "signer": int(row.participant_id),
                             "split": getattr(row, "split", None)})
    return examples, excluded


# ================================================================== batching
def pad_batch(examples, xs=None):
    """Right-pad features and labels; keep the true lengths.
    Padding values: features 0, labels 0 (ignored by CTC beyond label_length)."""
    xs = xs if xs is not None else [e["x"] for e in examples]
    x_len = np.array([len(x) for x in xs], np.int32)
    y_len = np.array([len(e["y"]) for e in examples], np.int32)
    X = np.zeros((len(xs), x_len.max(), NUM_FEATURES), np.float32)
    Y = np.zeros((len(xs), max(1, y_len.max())), np.int32)
    for i, (x, e) in enumerate(zip(xs, examples)):
        X[i, :len(x)] = x
        Y[i, :len(e["y"])] = e["y"]
    return X, x_len, Y, y_len


def iterate_batches(examples, batch_size, normalizer, rng=None, augment_cfg=None):
    """Length-bucketed batches (similar lengths together = less padding).
    Shuffled when rng is given. Augmentation only if augment_cfg is given."""
    order = np.argsort([len(e["x"]) for e in examples], kind="stable")
    batches = [order[i:i + batch_size] for i in range(0, len(order), batch_size)]
    if rng is not None:
        rng.shuffle(batches)
    for idx in batches:
        exs = [examples[i] for i in idx]
        xs = [e["x"] for e in exs]
        if augment_cfg:
            xs = [augment(x, vocab.required_frames(e["y"].tolist()), rng, augment_cfg)
                  for x, e in zip(xs, exs)]
        xs = [normalizer.apply(x) for x in xs]
        yield pad_batch(exs, xs)


DEFAULT_AUGMENT = {"stretch": (0.8, 1.25), "rotate_deg": 10.0, "scale": (0.9, 1.1),
                   "aspect": (0.9, 1.1), "wrist_shift": 0.05, "noise": 0.01, "drop_frame": 0.03}


def augment(x, min_len, rng, cfg=DEFAULT_AUGMENT):
    """Training-only random changes to UNnormalised features (T, 68).
    Never shortens a sequence below what its transcript needs (min_len)."""
    x = x.copy()
    T = len(x)
    lo, hi = cfg["stretch"]
    r = rng.uniform(lo, hi)
    new_T = int(round(T * r))
    if new_T >= max(min_len, 1) and new_T != T:
        idx = np.minimum((np.arange(new_T) / r).astype(int), T - 1)
        x = x[idx]
        x[:, WRIST_VEL] /= r                      # faster playback = larger steps per frame
    valid = x[:, VALID] > 0.5
    f = x[:, FINGERS].reshape(len(x), 21, 3)
    a = np.deg2rad(rng.uniform(-cfg["rotate_deg"], cfg["rotate_deg"]))
    rot = np.array([[np.cos(a), -np.sin(a)], [np.sin(a), np.cos(a)]], np.float32)
    f[..., :2] = f[..., :2] @ rot.T
    f *= rng.uniform(*cfg["scale"])
    f[..., 0] *= rng.uniform(*cfg["aspect"])
    f += rng.normal(0, cfg["noise"], f.shape).astype(np.float32)
    f[:, 0] = 0.0                                 # wrist stays the origin
    x[:, FINGERS] = f.reshape(len(x), 63)
    x[:, WRIST_POS] += rng.uniform(-cfg["wrist_shift"], cfg["wrist_shift"], 2).astype(np.float32)
    drop = rng.random(len(x)) < cfg["drop_frame"]
    x[~valid | drop] = 0.0                        # invalid frames stay all-zero
    return x
''',
    "asl/seq/train.py": r'''"""
train.py - Explicit CTC training loop with checkpoints that survive Colab
disconnects.

Why a hand-written loop instead of model.fit()?  CTC needs FOUR inputs per
batch (features, their true lengths, labels, label lengths). Writing the
step out makes it obvious which lengths go where:

    logits = model(features)                       (batch, T, 27) raw scores
    loss   = tf.nn.ctc_loss(labels, logits,
                            label_length = true letters per example,
                            logit_length = true frames per example,
                            blank_index  = 0)
    gradients clipped by global norm, then Adam.

Two kinds of saved output (both in the run folder, on Google Drive):
    checkpoints/last/   weights + optimizer state + progress  -> resume training
    best_model/         inference bundle of the best validation CER -> the laptop
"""
import json
import platform
import random
import shutil
import time
from pathlib import Path

import numpy as np

from . import data as sdata
from .ctc import greedy_decode, character_error_rate
from .model import save_inference_bundle
from .vocab import decode_ids

DEFAULT_TRAIN_CONFIG = {
    "epochs": 60,
    "batch_size": 32,
    "learning_rate": 1e-3,
    "clip_norm": 1.0,
    "seed": 42,
    "augment": True,
    "patience": 15,            # stop if validation CER has not improved for this many epochs
    "show_examples": 5,
    "log_every": 1,            # print every N-th epoch (best epochs are always printed)
}


def set_seeds(seed):
    import tensorflow as tf

    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)


def environment_versions():
    import tensorflow as tf

    v = {"python": platform.python_version(), "platform": platform.platform(),
         "tensorflow": tf.__version__, "keras": tf.keras.__version__ if hasattr(tf.keras, "__version__") else "",
         "numpy": np.__version__,
         "gpus": [d.name for d in tf.config.list_physical_devices("GPU")]}
    try:
        import keras
        v["keras"] = keras.__version__
    except Exception:
        pass
    for name in ("pandas", "pyarrow", "kaggle"):
        try:
            v[name] = __import__(name).__version__
        except Exception:
            pass
    return v


class CTCTrainer:
    def __init__(self, model, learning_rate=1e-3, clip_norm=1.0):
        import tensorflow as tf

        self.tf = tf
        self.model = model
        self.clip_norm = float(clip_norm)
        self.optimizer = tf.keras.optimizers.Adam(learning_rate)
        self.optimizer.build(model.trainable_variables)
        sig = [tf.TensorSpec([None, None, model.input_shape[-1]], tf.float32),
               tf.TensorSpec([None], tf.int32),
               tf.TensorSpec([None, None], tf.int32),
               tf.TensorSpec([None], tf.int32)]
        self._train_step = tf.function(self._train_step_impl, input_signature=sig)
        self._eval_step = tf.function(self._eval_step_impl, input_signature=sig)
        self._logits = tf.function(lambda x: model(x, training=False), input_signature=sig[:1])

    def ctc_loss(self, logits, x_len, y, y_len):
        """Mean CTC loss per letter (dividing by label length keeps long and
        short words on a similar scale)."""
        tf = self.tf
        loss = tf.nn.ctc_loss(labels=y, logits=logits, label_length=y_len, logit_length=x_len,
                              logits_time_major=False, blank_index=0)
        return tf.reduce_mean(loss / tf.cast(y_len, tf.float32))

    def _train_step_impl(self, x, x_len, y, y_len):
        tf = self.tf
        with tf.GradientTape() as tape:
            logits = self.model(x, training=True)
            loss = self.ctc_loss(logits, x_len, y, y_len)
        grads = tape.gradient(loss, self.model.trainable_variables)
        grads, norm = tf.clip_by_global_norm(grads, self.clip_norm)
        self.optimizer.apply_gradients(zip(grads, self.model.trainable_variables))
        return loss, norm

    def _eval_step_impl(self, x, x_len, y, y_len):
        logits = self.model(x, training=False)
        return self.ctc_loss(logits, x_len, y, y_len), logits

    def train_step(self, X, x_len, Y, y_len):
        loss, norm = self._train_step(X, x_len, Y, y_len)
        return float(loss), float(norm)

    def logits(self, X):
        return self._logits(X).numpy()

    # ------------------------------------------------------- checkpoints
    def save(self, folder, state):
        """Atomically replace folder with weights + optimizer + state.json."""
        folder = Path(folder)
        tmp = folder.with_name(folder.name + ".tmp")
        shutil.rmtree(tmp, ignore_errors=True)
        tmp.mkdir(parents=True)
        self.model.save_weights(str(tmp / "model.weights.h5"))
        np.savez(tmp / "optimizer.npz", *[v.numpy() for v in self.optimizer.variables])
        (tmp / "state.json").write_text(json.dumps(state, indent=2))
        old = folder.with_name(folder.name + ".old")
        shutil.rmtree(old, ignore_errors=True)
        if folder.exists():
            folder.rename(old)
        tmp.rename(folder)
        shutil.rmtree(old, ignore_errors=True)

    def restore(self, folder):
        folder = Path(folder)
        if not (folder / "state.json").exists():
            old = folder.with_name(folder.name + ".old")   # crash between the two renames
            if (old / "state.json").exists():
                folder = old
            else:
                return None
        self.model.load_weights(str(folder / "model.weights.h5"))
        z = np.load(folder / "optimizer.npz")
        values = [z[f"arr_{i}"] for i in range(len(z.files))]
        if len(values) != len(self.optimizer.variables):
            raise ValueError("Optimizer state does not match this model/optimizer")
        for var, val in zip(self.optimizer.variables, values):
            var.assign(val)
        return json.loads((folder / "state.json").read_text())


def predict_examples(trainer, examples, normalizer, batch_size=64):
    """Greedy-decoded text for each example (in the given order) + logits."""
    texts, all_logits = [], []
    for i in range(0, len(examples), batch_size):
        exs = examples[i:i + batch_size]
        X, x_len, _, _ = sdata.pad_batch(exs, [normalizer.apply(e["x"]) for e in exs])
        L = trainer.logits(X)
        for k, n in enumerate(x_len):
            all_logits.append(L[k, :n])
            texts.append(decode_ids(greedy_decode(L[k], n)))
    return texts, all_logits


def validate(trainer, examples, normalizer, batch_size=64):
    losses = []
    for X, x_len, Y, y_len in sdata.iterate_batches(examples, batch_size, normalizer):
        loss, _ = trainer._eval_step(X, x_len, Y, y_len)
        losses.append((float(loss), len(X)))
    hyps, _ = predict_examples(trainer, examples, normalizer, batch_size)
    refs = [e["text"] for e in examples]
    return {"loss": sum(l * n for l, n in losses) / max(1, sum(n for _, n in losses)),
            "cer": character_error_rate(refs, hyps),
            "word_acc": float(np.mean([r == h for r, h in zip(refs, hyps)])) if refs else 0.0,
            "pairs": list(zip(refs, hyps))}


def fit(model, train_examples, val_examples, normalizer, run_dir, preprocessing,
        model_cfg, cfg=None, log=print, versions=None):
    """Train with resume support. Safe to call again after a disconnect:
    it continues from run_dir/checkpoints/last if that exists."""
    cfg = dict(DEFAULT_TRAIN_CONFIG, **(cfg or {}))
    run_dir = Path(run_dir)
    run_dir.mkdir(parents=True, exist_ok=True)
    (run_dir / "train_config.json").write_text(json.dumps(cfg, indent=2))
    set_seeds(cfg["seed"])
    trainer = CTCTrainer(model, cfg["learning_rate"], cfg["clip_norm"])
    last = run_dir / "checkpoints" / "last"
    state = trainer.restore(last) or {"epoch": 0, "step": 0, "best_cer": None,
                                      "best_epoch": None, "history": []}
    if state["epoch"]:
        log(f"Resumed from epoch {state['epoch']} (best val CER {state['best_cer']})")
    aug = (sdata.DEFAULT_AUGMENT if cfg["augment"] is True else cfg["augment"]) or None

    for epoch in range(state["epoch"] + 1, cfg["epochs"] + 1):
        t0 = time.time()
        rng = np.random.default_rng(cfg["seed"] * 1000 + epoch)   # same order after a resume
        losses = []
        for X, x_len, Y, y_len in sdata.iterate_batches(train_examples, cfg["batch_size"],
                                                        normalizer, rng, aug):
            loss, _ = trainer.train_step(X, x_len, Y, y_len)
            if not np.isfinite(loss):
                raise FloatingPointError(f"Loss became {loss} at epoch {epoch}; check data/lengths")
            losses.append(loss)
            state["step"] += 1
        v = validate(trainer, val_examples, normalizer)
        row = {"epoch": epoch, "train_loss": float(np.mean(losses)), "val_loss": v["loss"],
               "val_cer": v["cer"], "val_word_acc": v["word_acc"], "seconds": round(time.time() - t0, 1)}
        state["history"].append(row)
        improved = state["best_cer"] is None or v["cer"] < state["best_cer"]
        if improved:
            state["best_cer"], state["best_epoch"] = v["cer"], epoch
            save_inference_bundle(run_dir / "best_model", model, model_cfg, preprocessing,
                                  versions=versions,
                                  reference_inputs=[normalizer.apply(e["x"]) for e in val_examples[:3]])
        state["epoch"] = epoch
        trainer.save(last, state)
        if improved or epoch % cfg["log_every"] == 0 or epoch == cfg["epochs"]:
            log(f"epoch {epoch:3d}  train loss {row['train_loss']:.3f}  val loss {v['loss']:.3f}  "
                f"val CER {v['cer']:.3f}  word acc {v['word_acc']:.3f}  ({row['seconds']}s)"
                + ("  * best, saved" if improved else ""))
            for ref, hyp in v["pairs"][:cfg["show_examples"]]:
                log(f"      true {ref:<16s} predicted {hyp}")
        if epoch - state["best_epoch"] >= cfg["patience"]:
            log(f"Stopping: no improvement for {cfg['patience']} epochs")
            break
    (run_dir / "history.json").write_text(json.dumps(state["history"], indent=2))
    return trainer, state
''',
    "asl/seq/evaluate.py": r'''"""
evaluate.py - Evaluation report for the sequence model.

All numbers come from the split you pass in (normally the TEST signers, who
were never used for training or model selection). They describe THIS
dataset subset only and must not be compared directly with numbers from
other datasets, other subsets, or the baseline's single-photo accuracy.

Report contents
    cer, word_acc              character error rate, exact word accuracy
    errors                     substitutions / insertions / deletions
    double_letters             words with a repeated letter (LL in HELLO):
                               accuracy, and how often the double was
                               collapsed to one letter
    by_speed                   slow / normal / fast, measured as frames per
                               letter (edges = training-split tertiles)
    per_signer                 CER of each unfamiliar signer
    idle                       letters emitted on inputs with NO hand at all
                               (a proxy only; real idle / transition
                               behaviour needs local recordings)
"""
import json
from pathlib import Path

import numpy as np

from .ctc import (character_error_rate, edit_operations, greedy_decode,
                  has_double_letter, collapse_doubles)
from .features import NUM_FEATURES
from .vocab import decode_ids


def speed_edges(train_examples):
    fpl = [len(e["x"]) / len(e["y"]) for e in train_examples]
    return [float(np.percentile(fpl, 33.3)), float(np.percentile(fpl, 66.7))]


def _summary(refs, hyps):
    if not refs:
        return {"n": 0}
    s = i = d = 0
    for r, h in zip(refs, hyps):
        a, b, c = edit_operations(r, h)
        s, i, d = s + a, i + b, d + c
    return {"n": len(refs), "cer": round(character_error_rate(refs, hyps), 4),
            "word_acc": round(float(np.mean([r == h for r, h in zip(refs, hyps)])), 4),
            "substitutions": s, "insertions": i, "deletions": d}


def idle_insertions(trainer, normalizer, seconds=(1, 3, 10), fps=30.0):
    """Letters the model emits for 'no hand visible' input of a few lengths."""
    out = {}
    for s in seconds:
        x = normalizer.apply(np.zeros((int(s * fps), NUM_FEATURES), np.float32))
        logits = trainer.logits(x[None])[0]
        out[f"{s}s_no_hand"] = decode_ids(greedy_decode(logits))
    return out


def evaluate(trainer, examples, normalizer, train_examples, split_name="test",
             fps=30.0, n_samples=30):
    from .train import predict_examples

    hyps, _ = predict_examples(trainer, examples, normalizer)
    refs = [e["text"] for e in examples]
    report = {"split": split_name, "overall": _summary(refs, hyps)}

    dbl = [k for k, r in enumerate(refs) if has_double_letter(r)]
    report["double_letters"] = dict(
        _summary([refs[k] for k in dbl], [hyps[k] for k in dbl]),
        collapsed_to_single=int(sum(hyps[k] == collapse_doubles(refs[k]) for k in dbl)))

    edges = speed_edges(train_examples)
    buckets = {"slow": [], "normal": [], "fast": []}
    for k, e in enumerate(examples):
        fpl = len(e["x"]) / len(e["y"])            # more frames per letter = slower signing
        name = "fast" if fpl < edges[0] else "normal" if fpl < edges[1] else "slow"
        buckets[name].append(k)
    report["by_speed"] = {"frames_per_letter_edges": edges,
                          **{n: _summary([refs[k] for k in ix], [hyps[k] for k in ix])
                             for n, ix in buckets.items()}}

    signers = sorted({e["signer"] for e in examples})
    report["per_signer"] = {str(s): _summary(*zip(*[(refs[k], hyps[k]) for k, e in enumerate(examples)
                                                    if e["signer"] == s]))
                            for s in signers}
    report["idle"] = idle_insertions(trainer, normalizer, fps=fps)
    report["samples"] = [{"true": r, "predicted": h} for r, h in zip(refs[:n_samples], hyps[:n_samples])]
    report["note"] = ("Measured on a pilot subset of Google's ASL Fingerspelling dataset, "
                      "signer-independent split. Not comparable to other datasets. "
                      "Laptop latency and real idle/transition behaviour are measured separately.")
    return report


def save_report(report, path):
    Path(path).write_text(json.dumps(report, indent=2))


def print_report(report):
    o = report["overall"]
    print(f"[{report['split']}] {o['n']} recordings  CER {o.get('cer')}  word accuracy {o.get('word_acc')}")
    print(f"   errors: {o.get('substitutions')} substituted, {o.get('insertions')} inserted, "
          f"{o.get('deletions')} deleted letters")
    d = report["double_letters"]
    print(f"   double-letter words: {d['n']}  word accuracy {d.get('word_acc')}  "
          f"collapsed to single letter {d['collapsed_to_single']}")
    for n in ("slow", "normal", "fast"):
        b = report["by_speed"][n]
        print(f"   {n:<6s}: {b['n']} recordings  CER {b.get('cer')}  word acc {b.get('word_acc')}")
    print(f"   letters emitted with no hand visible: {report['idle']}")
''',
}
for _name, _text in _FILES.items():
    _path = CODE / _name
    _path.parent.mkdir(parents=True, exist_ok=True)
    _path.write_text(_text, encoding="utf-8")
if str(CODE) not in sys.path:
    sys.path.insert(0, str(CODE))

import numpy as np
import matplotlib.pyplot as plt
from IPython.display import HTML, display
import asl.seq.vocab as vocab, asl.seq.features as features, asl.seq.fsboard as fb
import asl.seq.fsboard_video as fsv, asl.seq.model as smodel, asl.seq.train as strain
import asl.seq.evaluate as seval
for _m in (vocab, features, fb, fsv, smodel, strain, seval):
    importlib.reload(_m)

try:
    _found = fb.kaggle_auth_from_colab_secrets()
except ImportError:
    _found = []
print("Setup finished. Temporary working folder:", WORK)
print("Kaggle key from Colab Secrets:", ", ".join(_found) if _found else
      "none set (fine: FSboard downloaded without one when last tested)")

Setup finished. Temporary working folder: /content/fsboard_tcn
Kaggle key from Colab Secrets: none set (fine: FSboard downloaded without one when last tested)


---
## 1. Check the GPU and connect Google Drive

**In plain English.** A GPU makes training much faster. Google Drive keeps your work safe, because Colab's own disk
is erased when the session ends.

**What success looks like:** a line starting `GPU: [PhysicalDevice`, `Mounted at /content/drive`, and your free space.
If it says `GPU: NONE`, use **Runtime → Change runtime type → T4 GPU**. A CPU also works, just more slowly.

`RESUME_RUN` is only for new sessions. Empty means "the data step 4 saved last". To use an older run, put its
folder name from `runs/` here.

In [ ]:
import platform
import tensorflow as tf

gpus = tf.config.list_physical_devices("GPU")
print("Python", platform.python_version(), "| TensorFlow", tf.__version__)
print("GPU:", gpus if gpus else "NONE (Runtime > Change runtime type > T4 GPU)")

if os.environ.get("FSBOARD_DRIVE"):
    DRIVE = Path(os.environ["FSBOARD_DRIVE"])
else:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE = Path("/content/drive/MyDrive/asl_fingerspelling/fsboard_tcn_ctc")
SEQUENCES = DRIVE / "sequences"
SEQUENCES.mkdir(parents=True, exist_ok=True)
print("Saved work goes to:", DRIVE)
print(f"Free space: Colab disk {fb.free_gb(WORK):.0f} GB, Drive {fb.free_gb(DRIVE):.1f} GB")

# ---- Save and reload the prepared data, so steps 5-10 work in a new session without steps 2-4 ----
RESUME_RUN = ""  #@param {type:"string"}


def _plain(value):
    return value.item() if hasattr(value, "item") else value


def save_prepared():
    # End of step 4: write the model inputs and their settings to Drive as one file.
    base = RUN_NAME.split("__")[0]
    rows = [(split, e) for split in fsv.SPLITS for e in examples[split]]
    info = {"run": base, "seed": SEED, "feature_cfg": FEATURE_CFG, "prep": PREP, "unfamiliar": unfamiliar,
            "examples": [dict({k: _plain(v) for k, v in e.items() if k not in ("x", "y")}, group=split)
                         for split, e in rows]}
    local = WORK / "prepared.npz"
    np.savez_compressed(local, x=np.concatenate([e["x"] for _, e in rows]),
                        x_lengths=np.array([len(e["x"]) for _, e in rows], np.int64),
                        y=np.concatenate([np.asarray(e["y"]) for _, e in rows]),
                        y_lengths=np.array([len(e["y"]) for _, e in rows], np.int64),
                        info=np.array(json.dumps(info)))
    size_gb = local.stat().st_size / 1e9
    if fb.free_gb(DRIVE) < size_gb + 0.2:
        print(f"NOT saved for later sessions: the file is {size_gb:.2f} GB and Drive has "
              f"{fb.free_gb(DRIVE):.1f} GB free. This session still works; free some Drive space and run step 4 again.")
        return
    target = DRIVE / "runs" / base / "prepared.npz"
    target.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy(local, target)
    (DRIVE / "last_run.json").write_text(json.dumps({"run": base}))
    print(f"Saved for later sessions: {target} ({size_gb * 1000:.0f} MB, {len(rows)} clips). "
          "In a new session, run Setup and step 1, then go straight to step 5 or later.")


def need_data(trained=False):
    # First line of steps 5-10. Does nothing if this session already has the data;
    # otherwise loads what step 4 saved. trained=True also finds the trained model's folder.
    g = globals()
    if not all(k in g for k in ("examples", "normalizer", "PREP", "RUN_NAME", "RUN_DIR", "SEED", "unfamiliar")):
        pointer = DRIVE / "last_run.json"
        base = RESUME_RUN.split("__")[0] or (json.loads(pointer.read_text())["run"] if pointer.exists() else "")
        saved = DRIVE / "runs" / base / "prepared.npz"
        if not base or not saved.exists():
            raise FileNotFoundError(
                "This session has no prepared data, and none is saved on Drive"
                + (f" for the run '{base}'" if base else "") + ".\n"
                "What to do: run steps 2, 3 and 4 once (a CPU runtime is enough). Step 4 saves the result, "
                "and later sessions then only need Setup and step 1.")
        local = WORK / "prepared_loaded.npz"
        shutil.copy(saved, local)
        with np.load(local, allow_pickle=False) as z:
            info = json.loads(str(z["info"]))
            x, y = z["x"], z["y"]
            x_ends, y_ends = np.cumsum(z["x_lengths"]), np.cumsum(z["y_lengths"])
            x_lengths, y_lengths = z["x_lengths"], z["y_lengths"]
        loaded = {split: [] for split in fsv.SPLITS}
        for i, row in enumerate(info["examples"]):
            e = dict(row)
            e["x"] = x[x_ends[i] - x_lengths[i]:x_ends[i]]
            e["y"] = y[y_ends[i] - y_lengths[i]:y_ends[i]]
            loaded[e.pop("group")].append(e)
        g.update(examples=loaded, PREP=info["prep"], FEATURE_CFG=info["feature_cfg"], SEED=info["seed"],
                 unfamiliar=info["unfamiliar"], normalizer=features.Normalizer.from_dict(info["prep"]["normalizer"]),
                 MODEL_CFG=dict(smodel.DEFAULT_MODEL_CONFIG), RUN_NAME=info["run"],
                 RUN_DIR=DRIVE / "runs" / info["run"])
        print(f"Loaded the prepared data of run '{info['run']}' from Drive:",
              ", ".join(f"{len(loaded[s])} {s}" for s in fsv.SPLITS), "clips.")
        print("   If you changed step 2 since then, run steps 2, 3 and 4 again instead.")
    if trained and not (g["RUN_DIR"] / "best_model" / "model.weights.h5").exists():
        base = g["RUN_NAME"].split("__")[0]
        asked = DRIVE / "runs" / RESUME_RUN / "best_model" / "model.weights.h5"
        if "__" in RESUME_RUN and RESUME_RUN.split("__")[0] == base and asked.exists():
            found = [asked]
        else:
            found = sorted((p for p in (DRIVE / "runs").glob(base + "__*/best_model/model.weights.h5")
                            if not p.parents[1].name.endswith("__personal")), key=lambda p: p.stat().st_mtime)
        if not found:
            raise FileNotFoundError(f"No trained model for the run '{base}' on Drive yet.\n"
                                    "What to do: run step 6 first.")
        g.update(RUN_DIR=found[-1].parents[1], RUN_NAME=found[-1].parents[1].name)
        print(f"Using the trained model of run '{g['RUN_NAME']}' (the newest for this data). "
              "For another one, put its run name in RESUME_RUN in step 1 and run step 1 again.")

Python 3.13.16 | TensorFlow 2.21.0
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


MessageError: Error: credential propagation was unsuccessful

---
## 2. Choose your data

**In plain English.** Each clip has a phrase, a signer and an official group: **train**, **validation** or **test**.
Only metadata is downloaded here (about 70 MB). This shows what you would get **before** any video is downloaded.

**Two kinds of clips:**
* **One-word clips** (letters only), exactly as in your earlier runs. `0` means all of them, so the same 131 test
  clips can still be compared with earlier models.
* **Phrases with spaces**, such as `new york`, up to the `PHRASE_CLIPS_*` limits. Only single spaces between words.

**What you can change:**
* `CLIPS_TRAIN`, `CLIPS_VALIDATION`, `CLIPS_TEST`: how many one-word clips (`0` = all).
* `INCLUDE_PHRASES` and `PHRASE_CLIPS_*`: how many phrases. More phrases take longer in step 3. Clips already
  extracted are skipped, so you can start small and raise the limits later.
* `MAX_LETTERS`, `MAX_PHRASE_LETTERS`: the longest word and phrase to accept (letters only, spaces not counted).

**Skipped whole:** anything with a digit or symbol, such as `2067 pso costa`, because the hands also spell those and
the model has no outputs for them. Nothing is ever cut out of a phrase.

**Validation and test** use only people who are **not** in the training metadata, so the results describe unfamiliar signers.

**What success looks like:** a table of clips per group, a few example phrases, and the run folder's name.

In [ ]:
EXTRA_TRAIN_CLIPS = 3000
WEAK_LETTERS = "QXYUZP"

def clip_score(text):
    # text is the phrase in capitals, for example "NEW YORK".
    # Return a number: higher = picked earlier, 0 = never picked.
    score = sum(text.count(letter) for letter in WEAK_LETTERS)       # 1 point for every weak letter
    if any(a == b != " " for a, b in zip(text, text[1:])):           # 1 point if it has a double letter
        score += 1
    return score

In [ ]:
CLIPS_TRAIN = 0                 #@param {type:"integer"}
CLIPS_VALIDATION = 0            #@param {type:"integer"}
CLIPS_TEST = 0                  #@param {type:"integer"}
MIN_LETTERS = 2                 #@param {type:"integer"}
MAX_LETTERS = 100               #@param {type:"integer"}
INCLUDE_PHRASES = True          #@param {type:"boolean"}
PHRASE_CLIPS_TRAIN = 8000       #@param {type:"integer"}
PHRASE_CLIPS_VALIDATION = 2000   #@param {type:"integer"}
PHRASE_CLIPS_TEST = 2000        #@param {type:"integer"}
MAX_PHRASE_LETTERS = 40         #@param {type:"integer"}
SEED = 42                       #@param {type:"integer"}
SKIP_SENSITIVE = False           #@param {type:"boolean"}
EXTRA_TRAIN_CLIPS = 3000
WEAK_LETTERS = "QXYUZP"

def clip_score(text):
    # text is the phrase in capitals, for example "NEW YORK".
    # Return a number: higher = picked earlier, 0 = never picked.
    score = sum(text.count(letter) for letter in WEAK_LETTERS)       # 1 point for every weak letter
    if any(a == b != " " for a, b in zip(text, text[1:])):           # 1 point if it has a double letter
        score += 1
    return score

wanted = {"train": CLIPS_TRAIN, "validation": CLIPS_VALIDATION, "test": CLIPS_TEST}
wanted_phrases = {"train": PHRASE_CLIPS_TRAIN, "validation": PHRASE_CLIPS_VALIDATION, "test": PHRASE_CLIPS_TEST}
limits = dict(min_letters=MIN_LETTERS, max_letters=MAX_LETTERS, skip_sensitive=SKIP_SENSITIVE)
phrase_limits = dict(min_letters=MIN_LETTERS, max_letters=MAX_PHRASE_LETTERS, skip_sensitive=SKIP_SENSITIVE,
                     phrases=True)
metadata = {}
for split in fsv.SPLITS:
    name = fsv.metadata_path(split)
    metadata[split] = fb.load_metadata(fb.download(name, WORK / "metadata" / Path(name).name,
                                                   expected_size=fb.remote_size(name)))

train_signers = {r["signerId"] for r in metadata["train"]}
clips, summary, unfamiliar = {}, [], {}
for split in fsv.SPLITS:
    rows, skipped = fsv.select_clips(metadata[split], split, **limits)
    available = len(rows)
    unfamiliar[split] = split != "train"
    if split != "train":
        kept, removed = fsv.drop_seen_signers(rows, train_signers)
        if kept:
            rows = kept
        else:                               # nothing left: use them, and say so in the report
            unfamiliar[split] = False
            print(f"WARNING: every usable {split} clip is from a training signer.")
    clips[split] = fsv.limit_clips(rows, wanted[split], SEED)
    one_word = len(clips[split])
    print(f"{split}: one-word clips skipped whole because:", dict(skipped.most_common()))

    phrase_rows, phrases_usable = [], 0
    if INCLUDE_PHRASES:
        found, phrase_skipped = fsv.select_clips(metadata[split], split, **phrase_limits)
        if split != "train":
            found, _ = fsv.drop_seen_signers(found, train_signers)   # evaluation: unfamiliar signers only
        phrases_usable = len(found)
        phrase_rows = fsv.limit_clips(found, wanted_phrases[split], SEED)
        clips[split] = clips[split] + phrase_rows
        print(f"{split}: phrases skipped whole because:", dict(phrase_skipped.most_common(6)))
    summary.append({"group": split, "clips in metadata": f"{len(metadata[split]):,}",
                    "one-word usable": available, "one-word chosen": one_word,
                    "phrases usable": phrases_usable, "phrases chosen": len(phrase_rows),
                    "signers": len({r["signer_id"] for r in clips[split]}),
                    "video download (GB)": round(sum(r["size_bytes"] for r in clips[split]) / 1e9, 2)})
display(HTML(fsv.html_table(summary)))
if not clips["train"] or not clips["validation"] or not clips["test"]:
    raise fb.FsboardError("A group has no clips.\nWhat to check: the limits at the top of this cell.")
print("Examples of chosen phrases:", [r["answer"] for r in clips["train"] if " " in r["answer"]][:8])

# Different data gets its own run folder, so these results never mix with earlier runs.
RUN_NAME = f"train{len(clips['train'])}_val{len(clips['validation'])}_test{len(clips['test'])}_letters{MIN_LETTERS}-{MAX_LETTERS}"
if INCLUDE_PHRASES:
    RUN_NAME += f"_phrases{MAX_PHRASE_LETTERS}"
RUN_DIR = DRIVE / "runs" / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)
(RUN_DIR / "selection.json").write_text(json.dumps({"seed": SEED, "limits": limits, "wanted": wanted,
                                                    "phrase_limits": phrase_limits if INCLUDE_PHRASES else None,
                                                    "wanted_phrases": wanted_phrases if INCLUDE_PHRASES else None,
                                                    "unfamiliar_signers": unfamiliar, "clips": clips}, indent=1))
print("Run folder:", RUN_DIR)

---
## 3. Extract all 21 hand points from the videos

**In plain English.** For each clip: download the video to temporary disk, check its fingerprint (MD5), turn it upright,
mirror it the way your webcam app does, run MediaPipe, save the small result to Drive, and **delete the video**.

MediaPipe gets its own small Python environment, because it needs older package versions than Colab's TensorFlow.
It is the same Python (3.11) and MediaPipe (0.10.14) as your laptop.

Safe to interrupt: finished clips are kept and skipped next time. One bad clip doesn't stop the run, but five failures
in a row do, because that usually means a connection or permission problem.

`DETECTION_CONFIDENCE` is how sure MediaPipe must be before accepting a hand. If step 4 shows few frames with a hand,
try a lower value such as `0.3`, but first delete `sequences/` from the Drive folder, or finished clips are skipped.

**What success looks like:** a line with the MediaPipe version, progress lines with minutes left, then `0 failed`. The
time depends on the number of clips and hasn't been measured on Colab yet; the progress lines estimate it.

In [ ]:
DETECTION_CONFIDENCE = 0.5  #@param {type:"number"}

os.environ["MPLBACKEND"] = "Agg"   # the MediaPipe helper must not inherit Colab's plotting backend
if os.environ.get("FSBOARD_MP_PYTHON"):
    mediapipe_python = os.environ["FSBOARD_MP_PYTHON"]
    print("MediaPipe:", fsv.check_mediapipe(mediapipe_python))
else:
    mediapipe_python = fsv.ensure_mediapipe_env(WORK / "mediapipe_env")

all_clips = [row for split in fsv.SPLITS for row in clips[split]]
report = fsv.run_extraction(all_clips, SEQUENCES, WORK / "videos", mediapipe_python, CODE,
                            detection=DETECTION_CONFIDENCE)
(RUN_DIR / "extraction_report.json").write_text(json.dumps(report, indent=1))
print(f"Done: {report['done']} new, {report['already_done']} already saved earlier, {len(report['failed'])} failed")
for failure in report["failed"][:10]:
    print("   ", failure["clip"], "-", failure["problem"].splitlines()[0][:120])
if report["stopped"]:
    raise fb.FsboardError(f"Extraction stopped after repeated failures.\n{report['stopped']}")
size_mb = sum(p.stat().st_size for p in SEQUENCES.rglob("*.npz")) / 1e6
print(f"Hand points on Drive: {size_mb:.1f} MB")

---
## 4. Turn the points into model inputs, and check them

**In plain English.** Each frame becomes 68 numbers:
* **63 finger numbers:** finger positions relative to the wrist, scaled to the hand's size.
* **Wrist position and speed:** so movement isn't lost.
* **A flag:** "a hand was found here".

Three more things happen:
* **One hand per clip:** a left hand is mirrored to look like a right hand.
* **Same speed:** every clip is converted to 30 frames per second using its real timestamps.
* **Same proportions:** hand shapes are corrected so a portrait phone video and a landscape webcam match.

The scaling numbers (mean and spread) are computed **only from training clips**, so test data can't leak in.

**What success looks like:** a table of examples per group, `failed checks: 0`, and a picture of a hand with **all five
fingers**. Clips with too few detected hand frames are left out, with the reason shown.

In [ ]:
FEATURE_CFG = dict(features.DEFAULT_CONFIG, **fsv.FEATURE_SETTINGS)
examples, table = {}, []
for split in fsv.SPLITS:
    examples[split], excluded = fsv.build_examples(clips[split], SEQUENCES, FEATURE_CFG)
    share = [e["hand_share"] for e in examples[split]]
    table.append({"group": split, "chosen": len(clips[split]), "usable": len(examples[split]),
                  "median share of frames with a hand": round(float(np.median(share)), 2) if share else "-",
                  "left out": dict(excluded) or "none"})
display(HTML(fsv.html_table(table)))
if not examples["train"] or not examples["validation"] or not examples["test"]:
    raise fb.FsboardError("A group has no usable clips.\nWhat to check: the extraction report in step 3, and use more clips.")

failed = 0
for split in fsv.SPLITS:
    for e in examples[split]:
        results = fsv.check_recording(fb.load_example(e["path"]))
        bad = [name for name, passed, _ in results if not passed]
        if bad or not np.all(np.isfinite(e["x"])) or e["x"].shape[1] != features.NUM_FEATURES:
            failed += 1
            print("Problem:", e["sequence_id"], bad)
print("failed checks:", failed)
overlap = {e["signer"] for e in examples["train"]} & {e["signer"] for s in ("validation", "test") for e in examples[s]}
print("signers in both training and evaluation:", len(overlap))

normalizer = features.Normalizer().fit([e["x"] for e in examples["train"]])
canonical = []
for e in examples["train"][:200]:
    rec = fb.load_example(e["path"])
    hand = fsv.signing_hand(rec)
    canonical.append(features.canonical_points(rec[f"{hand}_hand"], hand))
PREP = features.preprocessing_dict(normalizer, FEATURE_CFG, chirality_train=round(features.chirality_score(np.concatenate(canonical)), 3),
                                   source="FSboard videos, MediaPipe Hands 0.10.14, mirrored like the webcam app",
                                   vocab_version=vocab.VOCAB_VERSION)
(RUN_DIR / "preprocessing.json").write_text(json.dumps(PREP, indent=1))
MODEL_CFG = dict(smodel.DEFAULT_MODEL_CONFIG)

sample = max(examples["train"], key=lambda e: e["hand_share"])
fb.plot_example(fb.load_example(sample["path"]))
plt.show()
save_prepared()                           # later sessions load this instead of repeating steps 2-4

---
## 5. Optional tiny test: memorise a few clips

**In plain English.** The model trains and is tested on the **same 16 clips**, with no random changes. It should simply
memorise them. This only proves the pipeline works. It says nothing about real accuracy. It is off by default, because
the pipeline has already worked on the full data; tick `RUN_TINY_TEST` after changing the code.

**What success looks like:** the error rate (CER) falls from about 1.0 towards 0, and predicted words start to match.
If it stays near 1.0 or the loss becomes `nan`, don't continue: something in the pipeline is wrong.

In [ ]:
RUN_TINY_TEST = False  #@param {type:"boolean"}

if RUN_TINY_TEST:
    need_data()                           # in a new session: loads what step 4 saved
    tiny_dir = RUN_DIR.parent / (RUN_NAME.split("__")[0] + "_tiny")
    shutil.rmtree(tiny_dir, ignore_errors=True)
    tiny = examples["train"][:16]
    tf.keras.utils.set_random_seed(SEED)
    _, tiny_state = strain.fit(smodel.build_tcn_ctc(MODEL_CFG), tiny, tiny, normalizer, tiny_dir, PREP, MODEL_CFG,
                               cfg={"epochs": 300, "batch_size": 4, "augment": False, "learning_rate": 3e-3,
                                    "patience": 300, "show_examples": 3, "log_every": 25, "seed": SEED})
    print("Best error rate on the memorised clips:", round(tiny_state["best_cer"], 3))
else:
    print("Skipped.")

---
## 6. Train the model

**In plain English.** The model studies the training clips again and again. After each round (*epoch*) it is scored on
the validation signers, and you see a few true and predicted texts.

**The settings, and why:**
* `EXPERIMENT = "speed_long_memory"`: each output sees about 4 seconds (125 frames) instead of 2. It did best on
  slowed-down clips in your earlier runs. `"speed"` keeps the 2-second view.
* **Speed range:** every training clip is randomly played between 2× faster and 3.5× slower, so the model meets slow
  signing like yours.
* `WARM_START_FROM`: the run whose model this one starts from, instead of from zero. All its layers are copied; the
  output layer keeps its 27 letter outputs (blank and A–Z) and gets one new output for the **space**. So it starts
  out reading letters as well as that model and mainly has to learn spaces. Leave it empty to start from zero.
  It must be a `speed_long_memory` run when `EXPERIMENT` is `speed_long_memory` (the layers must match).

Saved every epoch: `checkpoints/last/`, so training continues after a disconnect, and `best_model/`, the version with
the lowest validation error. Each `EXPERIMENT` gets its own run folder.

**What success looks like:** `Warm start from ...: copied ... layers, extended ['logits']`, then epochs where the
validation error falls, then two plots. A big gap between low training loss and high validation error means it's
memorising rather than learning.

In [ ]:
EXPERIMENT = "speed_long_memory"  #@param ["speed_long_memory", "speed"]
WARM_START_FROM = "train732_val103_test131_letters2-100__speed_long_memory"  #@param {type:"string"}
EPOCHS = 80        #@param {type:"integer"}
BATCH_SIZE = 16    #@param {type:"integer"}
PATIENCE = 10      #@param {type:"integer"}


need_data()                               # in a new session: loads what step 4 saved
RUN_NAME = RUN_NAME.split("__")[0] + "__" + EXPERIMENT   # its own folder, so nothing resumes or overwrites
RUN_DIR = DRIVE / "runs" / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)
(RUN_DIR / "preprocessing.json").write_text(json.dumps(PREP, indent=1))
MODEL_CFG = dict(smodel.DEFAULT_MODEL_CONFIG)
if EXPERIMENT == "speed_long_memory":
    MODEL_CFG["dilations"] = [1, 2, 4, 8, 16]              # sees about 4 seconds instead of 2
SPEED_AUGMENT = dict(strain.sdata.DEFAULT_AUGMENT, stretch=(0.5, 3.5))   # 2x faster to 3.5x slower
print("Run:", RUN_NAME, "| the model sees", smodel.receptive_field(MODEL_CFG), "frames |",
      MODEL_CFG["num_classes"], "outputs (blank, A-Z, space)")

VERSIONS = strain.environment_versions()
TRAIN_CFG = dict(strain.DEFAULT_TRAIN_CONFIG, epochs=EPOCHS, batch_size=BATCH_SIZE, patience=PATIENCE, seed=SEED,
                 augment=SPEED_AUGMENT)
tf.keras.utils.set_random_seed(SEED)
model = smodel.build_tcn_ctc(MODEL_CFG)
if WARM_START_FROM:
    source = DRIVE / "runs" / WARM_START_FROM / "best_model"
    if not source.exists():
        raise FileNotFoundError(f"No model to start from at {source}.\n"
                                "What to check: WARM_START_FROM, or leave it empty to start from zero.")
    warm = smodel.warm_start(model, source)
    print(f"Warm start from {WARM_START_FROM}: copied {len(warm['copied'])} layers, "
          f"extended {warm['extended']} with the space output.")
_, state = strain.fit(model, examples["train"], examples["validation"], normalizer, RUN_DIR, PREP, MODEL_CFG,
                      cfg=TRAIN_CFG, versions=VERSIONS)
history = state["history"]
fig, ax = plt.subplots(1, 2, figsize=(10, 3))
epochs = [h["epoch"] for h in history]
ax[0].plot(epochs, [h["train_loss"] for h in history], label="training")
ax[0].plot(epochs, [h["val_loss"] for h in history], label="validation")
ax[0].set(title="Loss", xlabel="epoch"); ax[0].legend()
ax[1].plot(epochs, [h["val_cer"] for h in history])
ax[1].set(title="Validation character error rate", xlabel="epoch")
plt.show()
print("Best validation error rate", round(state["best_cer"], 3), "at epoch", state["best_epoch"])

---
## 7. Evaluate, next to your earlier model

**In plain English.** This scores the **best saved model** on signers it never trained on:

| Measure | Meaning |
|---|---|
| **Character error rate (CER)** | wrong letters and spaces per true character; lower is better |
| **Letters-only error rate** | the same with spaces ignored, so it compares fairly with models that have no space |
| **Word error rate** | wrong, missing or extra words per true word |
| **Spaces found / extra spaces** | how many true spaces it wrote in the right place, and how many it invented |
| **Exact accuracy** | texts that are completely right |
| **Double letters** | words like HELLO, and how often LL collapsed to L |
| **Slow, normal, fast** | signing speed, measured as frames per letter |
| **Per letter** | which letters are weak, and what they get confused with |
| **No hand visible** | whether it invents letters when no hand is shown |

`COMPARE_WITH` is your earlier model. Both are scored on the **same test clips**, each with its own scaling numbers.
On phrases the earlier model can only be judged on letters, because it has no space output.

Look at the **test** numbers, and don't tune settings on them. They describe these clips only.

**What success looks like:** the tables below and a saved `evaluation_report.json`.

In [ ]:
COMPARE_WITH = "train8732_val2103_test2131_letters2-100_phrases40__speed_long_memory__personal"  #@param {type:"string"}


def load_for_scoring(folder):
    # A saved model and its own scaling numbers.
    scored, _, scored_prep = smodel.load_inference_bundle(folder)
    return strain.CTCTrainer(scored), features.Normalizer.from_dict(scored_prep["normalizer"])


def all_scores(refs, hyps):
    extra = {k: v for k, v in fsv.phrase_scores(refs, hyps).items()
             if k in ("letters_only_error_rate", "word_error_rate", "spaces_found", "extra_spaces")}
    return dict(fsv.score_texts(refs, hyps), **extra)


need_data(trained=True)                   # in a new session: loads the data and finds the trained model
evaluator, _ = load_for_scoring(RUN_DIR / "best_model")
reports = {}
for split in ("validation", "test"):
    group = examples[split]
    report = seval.evaluate(evaluator, group, normalizer, examples["train"], split,
                            fps=PREP["features"]["target_fps"])
    seval.print_report(report)
    hyps, _ = strain.predict_examples(evaluator, group, normalizer)
    refs = [e["text"] for e in group]
    report["scores"] = all_scores(refs, hyps)
    report["by_letter"] = fsv.letter_report(refs, hyps)
    report["signers_unfamiliar"] = unfamiliar[split]
    reports[split] = report

score_groups = {"one-word test clips": [e for e in examples["test"] if " " not in e["text"]],
                "phrase test clips": [e for e in examples["test"] if " " in e["text"]]}
score_models = {"this run": RUN_DIR / "best_model"}
compare_dir = DRIVE / "runs" / COMPARE_WITH / "best_model"
if COMPARE_WITH and COMPARE_WITH != RUN_NAME and compare_dir.exists():
    score_models["earlier model"] = compare_dir
elif COMPARE_WITH and COMPARE_WITH != RUN_NAME:
    print("No earlier model at", compare_dir, "- comparison skipped.")
comparison, compare_hyps = [], {}
for label, folder in score_models.items():
    scored_eval, scored_norm = load_for_scoring(folder)
    for group_name, items in score_groups.items():
        if items:
            hyps, _ = strain.predict_examples(scored_eval, items, scored_norm)
            compare_hyps[(label, group_name)] = hyps
            comparison.append(dict(model=label, group=group_name, **all_scores([e["text"] for e in items], hyps)))
reports["comparison"] = comparison

print("\nThis run next to the earlier model, on the same test clips (lower error rates are better):")
display(HTML(fsv.html_table(comparison)))
print("\nTest clips, weakest letters first:")
display(HTML(fsv.html_table(reports["test"]["by_letter"]["letters"])))
print("Most common confusions:")
display(HTML(fsv.html_table(reports["test"]["by_letter"]["confusions"])))
for group_name, items in score_groups.items():
    if items:
        print(f"Examples, {group_name}:")
        shown = [{"true": e["text"], "this run": h} for e, h in zip(items, compare_hyps[("this run", group_name)])]
        if ("earlier model", group_name) in compare_hyps:
            for row, h in zip(shown, compare_hyps[("earlier model", group_name)]):
                row["earlier model"] = h
        display(HTML(fsv.html_table(shown[:15])))
seval.save_report(reports, RUN_DIR / "best_model" / "evaluation_report.json")
print("Saved:", RUN_DIR / "best_model" / "evaluation_report.json")

---
## 8. Speed check: the same test clips, faster and slower

**In plain English.** The one-word test clips are played 2× faster, at normal speed, 2× slower and 3× slower (by
repeating or skipping frames), and scored again, for this run and the earlier model. Your own signing is about 3×
slower than FSboard's signers. This is a simulation: real slow signing moves differently, so your webcam is the real test.

**What success looks like:** a table with four rows per model. Flat numbers across the rows mean the model copes with
different speeds.

In [ ]:
RUN_SPEED_CHECK = True  #@param {type:"boolean"}


def at_speed(x, slower):
    # Features as if signed `slower` times slower (0.5 = twice as fast).
    n = max(1, round(len(x) * slower))
    y = x[np.minimum((np.arange(n) / slower).astype(int), len(x) - 1)].copy()
    y[:, features.WRIST_VEL] /= slower          # wrist speed changes with signing speed
    return y


need_data(trained=True)                   # in a new session: loads the data and finds the trained model
if "score_models" not in globals():       # step 7 was not run in this session: score this run only
    def load_for_scoring(folder):
        scored, _, scored_prep = smodel.load_inference_bundle(folder)
        return strain.CTCTrainer(scored), features.Normalizer.from_dict(scored_prep["normalizer"])
    score_groups = {"one-word test clips": [e for e in examples["test"] if " " not in e["text"]]}
    score_models = {"this run": RUN_DIR / "best_model"}

if RUN_SPEED_CHECK and score_groups["one-word test clips"]:
    speed_rows = []
    words = score_groups["one-word test clips"]
    for label, folder in score_models.items():
        scored_eval, scored_norm = load_for_scoring(folder)
        for signing, slower in (("2x faster", 0.5), ("normal", 1.0), ("2x slower", 2.0), ("3x slower", 3.0)):
            group = [dict(e, x=at_speed(e["x"], slower)) for e in words]
            hyps, _ = strain.predict_examples(scored_eval, group, scored_norm)
            speed_rows.append(dict(model=label, signing=signing, **fsv.score_texts([e["text"] for e in group], hyps)))
    display(HTML(fsv.html_table(speed_rows)))
    (RUN_DIR / "best_model" / "speed_report.json").write_text(json.dumps(speed_rows, indent=1))
else:
    print("Skipped.")

---
## 9. Export the small model file

**In plain English.** This makes one small zip on your Drive, holding the trained model, its settings and its reports.
Nothing comes to your laptop unless you tick `DOWNLOAD_TO_MY_COMPUTER`.

**On the laptop:** unzip it into a **new** folder, for example `backend\models\phrase_model`, so your other models stay
as they are. The folder must directly contain `model.weights.h5`. Then try it:
`venv\Scripts\python.exe scripts\sequence_model\seq_record_word.py backend\models\phrase_model --mirror-input`

**What success looks like:** a line naming a zip in `MyDrive/asl_fingerspelling/fsboard_tcn_ctc/export/`, under 1 MB.

In [ ]:
DOWNLOAD_TO_MY_COMPUTER = False  #@param {type:"boolean"}

need_data(trained=True)                   # in a new session: finds the trained model on Drive
bundle = RUN_DIR / "best_model"
for name in ("train_config.json", "history.json", "selection.json", "extraction_report.json"):
    for folder in (RUN_DIR, RUN_DIR.parent / RUN_NAME.split("__")[0]):
        if (folder / name).exists():
            shutil.copy(folder / name, bundle / name)
            break
(bundle / "README.txt").write_text(
    "TCN + CTC fingerspelling model (blank=0, A-Z, space), trained on FSboard videos with MediaPipe Hands.\n"
    f"Run: {RUN_NAME}. Dataset: FSboard by Georg et al., CC BY 4.0.\n"
    "Inputs use aspect_correction=True (fsboard_video.isotropic_points): live code must apply it too.\n")
(DRIVE / "export").mkdir(exist_ok=True)
export_zip = fb.zip_outputs(bundle, DRIVE / "export" / f"{RUN_NAME}.zip")
print(f"Saved: {export_zip} ({export_zip.stat().st_size / 1e6:.2f} MB)")
if DOWNLOAD_TO_MY_COMPUTER:
    from google.colab import files
    files.download(str(export_zip))

---
## 10. Optional: fine-tune on your own webcam recordings

**Off by default.** Use it only if the model from steps 6–9 reads your own signing poorly. Then:
1. On the laptop, record words and phrases with `seq_record_word.py --words ...` and pack them with
   `seq_pack_my_recordings.py` (see `docs/SEQUENCE_MODEL.md`).
2. Upload `backend/storage/my_recordings_pack.npz` to `MyDrive/asl_fingerspelling/fsboard_tcn_ctc/my_recordings/`.
3. Tick `FINE_TUNE_ON_MY_RECORDINGS` and run this cell. In a new Colab session it only needs Setup and step 1
   first (the data step 4 saved and the model from step 6 are read from Drive).

**What it does.** It continues training a **copy** of the model from step 6 (the model's own folder is never changed),
on all FSboard training clips plus your recordings, which are repeated so they make up about `YOUR_SHARE` of each
epoch. It uses a lower learning rate, so it adjusts gently. The best epoch is chosen on your validation takes; your
test takes, which are never trained on, give the honest score.

**What success looks like:** a before/after table. Keep the new model only if **your test words** improve and the
**FSboard** rows don't clearly get worse. On the laptop, unzip it into a new folder such as `backend\models\personal_phrase_model`.

In [ ]:
FINE_TUNE_ON_MY_RECORDINGS = True  #@param {type:"boolean"}
BASE_EXPERIMENT = "speed_long_memory"  #@param ["speed_long_memory", "speed"]
TUNE_EPOCHS = 40                    #@param {type:"integer"}
TUNE_PATIENCE = 20                  #@param {type:"integer"}
TUNE_LEARNING_RATE = 0.0003         #@param {type:"number"}
YOUR_SHARE = 0.5                    #@param {type:"number"}
DOWNLOAD_PERSONAL_MODEL = True     #@param {type:"boolean"}

if not FINE_TUNE_ON_MY_RECORDINGS:
    print("Skipped: tick FINE_TUNE_ON_MY_RECORDINGS to fine-tune on your own recordings.")
else:
    need_data()                           # in a new session: loads what step 4 saved
    BASE_RUN = RUN_NAME.split("__")[0] + "__" + BASE_EXPERIMENT   # the model trained in step 6
    BASE_DIR = DRIVE / "runs" / BASE_RUN
    PERSONAL_RUN = BASE_RUN + "__personal"
    PERSONAL_DIR = DRIVE / "runs" / PERSONAL_RUN
    PACK = DRIVE / "my_recordings" / "my_recordings_pack.npz"
    if not PACK.exists():
        raise FileNotFoundError(f"Upload my_recordings_pack.npz from your laptop to {PACK.parent}")
    if not (BASE_DIR / "best_model").exists():
        raise FileNotFoundError(f"No trained model at {BASE_DIR / 'best_model'}: run step 6 first, "
                                "or check BASE_EXPERIMENT")

    # 1. The model you start from, and the normaliser it was trained with (kept unchanged,
    #    so the laptop tools work with the new model exactly as before).
    base_model, base_cfg, base_prep = smodel.load_inference_bundle(BASE_DIR / "best_model")
    base_norm = features.Normalizer.from_dict(base_prep["normalizer"])

    # 2. Your recordings, packed on the laptop by seq_pack_my_recordings.py.
    with np.load(PACK, allow_pickle=False) as z:
        pack_cfg = json.loads(str(z["feature_cfg"]))
        ends = np.cumsum(z["lengths"])
        mine = [{"x": z["x"][e - n:e], "y": np.array(vocab.encode(str(w)), np.int32), "text": str(w),
                 "pace": str(p), "split": str(g), "sequence_id": str(f), "signer": "me"}
                for e, n, w, p, g, f in zip(ends, z["lengths"], z["words"], z["paces"], z["splits"], z["files"])]
    if any(" " in e["text"] for e in mine) and not smodel.has_space_output(base_cfg):
        raise ValueError("Your recordings include phrases with spaces, but this model has no space output. "
                         "Train with phrases in steps 2 and 6, or pack without the phrase lists.")
    if json.dumps(pack_cfg, sort_keys=True) != json.dumps(base_prep["features"], sort_keys=True):
        raise ValueError("Your recordings were packed with different feature settings than this model uses. "
                         "On the laptop, pack again with --bundle pointing at this model's folder.")
    mine_by = {s: [e for e in mine if e["split"] == s] for s in ("train", "validation", "test")}
    display(HTML(fsv.html_table([{"group": s, "takes": len(g), "different words": len({e["text"] for e in g})}
                                 for s, g in mine_by.items()])))
    if not mine_by["train"] or not mine_by["validation"]:
        raise ValueError("Fine-tuning needs your takes in both 'train' and 'validation'. Record more training words.")

    # 3. Training data: all FSboard training clips plus yours, repeated so they are about
    #    YOUR_SHARE of what the model sees. Keeping FSboard in stops it forgetting other signers.
    repeat = max(1, min(10, round(YOUR_SHARE / (1 - YOUR_SHARE) * len(examples["train"]) / len(mine_by["train"]))))
    train_set = examples["train"] + mine_by["train"] * repeat
    if max(int(np.max(e["y"])) for e in train_set) >= base_cfg["num_classes"]:
        raise ValueError("The training clips include spaces, but this model has no space output: check step 6.")
    print(f"Training on {len(examples['train'])} FSboard clips + {len(mine_by['train'])} of your takes x {repeat} "
          f"(yours are {len(mine_by['train']) * repeat / len(train_set):.0%} of each epoch).")
    print(f"The best epoch is chosen on your {len(mine_by['validation'])} validation takes; "
          f"your {len(mine_by['test'])} test takes are only used for the final score.")

    # 4. Continue training a copy of the base model, gently (lower learning rate), with the same
    #    random changes (augmentation) the base model was trained with. Safe to re-run after a
    #    disconnect: it resumes from PERSONAL_DIR/checkpoints/last.
    base_train_cfg = BASE_DIR / "train_config.json"
    augment = json.loads(base_train_cfg.read_text()).get("augment", True) if base_train_cfg.exists() else True
    personal_model = smodel.build_tcn_ctc(base_cfg)
    personal_model.set_weights(base_model.get_weights())
    PERSONAL_PREP = dict(base_prep, base_model=BASE_RUN,
                         fine_tuned_on="FSboard training clips + your own recordings")
    TUNE_CFG = dict(strain.DEFAULT_TRAIN_CONFIG, epochs=TUNE_EPOCHS, patience=TUNE_PATIENCE,
                    learning_rate=TUNE_LEARNING_RATE, augment=augment, seed=42)
    strain.fit(personal_model, train_set, mine_by["validation"], base_norm, PERSONAL_DIR, PERSONAL_PREP, base_cfg,
               cfg=TUNE_CFG, versions=strain.environment_versions())

    # 5. Before vs after, on the same takes. Your test words are the honest score: never trained on.
    after_model, _, _ = smodel.load_inference_bundle(PERSONAL_DIR / "best_model")
    test_words = [e for e in mine_by["test"] if " " not in e["text"]]
    groups = {"your test words": test_words}
    for pace in sorted({e["pace"] for e in test_words}):
        groups[f"your test words, {pace}"] = [e for e in test_words if e["pace"] == pace]
    groups["your test phrases"] = [e for e in mine_by["test"] if " " in e["text"]]
    groups["your validation takes"] = mine_by["validation"]
    groups["FSboard one-word test clips"] = [e for e in examples["test"] if " " not in e["text"]]
    groups["FSboard phrase test clips"] = [e for e in examples["test"] if " " in e["text"]]
    personal_rows, predictions = [], {}
    for label, m in (("before (base model)", base_model), ("after (personal)", after_model)):
        before_after = strain.CTCTrainer(m)
        for group, items in groups.items():
            if not items:
                continue
            hyps, _ = strain.predict_examples(before_after, items, base_norm)
            predictions[(label, group)] = hyps
            refs = [e["text"] for e in items]
            extra = {k: v for k, v in fsv.phrase_scores(refs, hyps).items()
                     if k in ("letters_only_error_rate", "word_error_rate", "spaces_found", "extra_spaces")}
            personal_rows.append(dict(model=label, group=group, **fsv.score_texts(refs, hyps), **extra))
    display(HTML(fsv.html_table(personal_rows)))
    for group in ("your test words", "your test phrases"):
        if groups[group]:
            print(f"{group.capitalize()} (first 20):")
            display(HTML(fsv.html_table([
                {"true": e["text"], "pace": e["pace"], "before": b, "after": a}
                for e, b, a in zip(groups[group], predictions[("before (base model)", group)],
                                   predictions[("after (personal)", group)])][:20])))

    # 6. Save the report and a small zip of the new model for the laptop.
    personal_bundle = PERSONAL_DIR / "best_model"
    for name in ("train_config.json", "history.json"):
        if (PERSONAL_DIR / name).exists():
            shutil.copy(PERSONAL_DIR / name, personal_bundle / name)
    (personal_bundle / "personal_report.json").write_text(json.dumps(personal_rows, indent=1))
    (personal_bundle / "README.txt").write_text(
        f"TCN + CTC fingerspelling model: {BASE_RUN} fine-tuned on FSboard + your own recordings.\n"
        "Inputs use aspect_correction=True (fsboard_video.isotropic_points): live code must apply it too.\n"
        "Contains no recordings, only model weights and settings.\n")
    (DRIVE / "export").mkdir(exist_ok=True)
    personal_zip = fb.zip_outputs(personal_bundle, DRIVE / "export" / f"{PERSONAL_RUN}.zip")
    print(f"Saved: {personal_zip} ({personal_zip.stat().st_size / 1e6:.2f} MB). "
          "On the laptop, unzip it into a new folder in backend\\models")
    if DOWNLOAD_PERSONAL_MODEL:
        from google.colab import files
        files.download(str(personal_zip))